In [1]:
# ============================================================================
# SIH26053 — 08 POINTNET++ STRONGER MODEL
# CELL 1 — PROJECT SETUP
# ============================================================================

import os
import json
import random
import time
import shutil
import platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/"
    "fovealidar_nuscenes_v1"
)

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"

# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------

ROOT_DIR = Path("/kaggle/working/strong_model")

CHECKPOINT_DIR = ROOT_DIR / "checkpoints"
RESULTS_DIR = ROOT_DIR / "results"
PREDICTIONS_DIR = ROOT_DIR / "predictions"
EXPORT_DIR = ROOT_DIR / "export"

for d in [
    ROOT_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
    PREDICTIONS_DIR,
    EXPORT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

MODEL_NAME = "pointnet_pp_strong"

NUM_CLASSES = 4
IGNORE_LABEL = 255

INPUT_FEATURES = [
    "x",
    "y",
    "z",
    "intensity",
    "range",
]

INPUT_CHANNELS = 5

NUM_EPOCHS = 50
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ---------------------------------------------------------------------------
# Environment information
# ---------------------------------------------------------------------------

print("=" * 75)
print("SIH26053 — 08 POINTNET++ STRONGER MODEL")
print("=" * 75)

print()
print("Dataset:")
print(f"  Path       : {DATASET_DIR}")
print(f"  Exists     : {DATASET_DIR.exists()}")

print()
print("Model:")
print(f"  Name       : {MODEL_NAME}")
print(f"  Input      : {INPUT_FEATURES}")
print(f"  Channels   : {INPUT_CHANNELS}")
print(f"  Classes    : {NUM_CLASSES}")

print()
print("Training:")
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  LR         : {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Seed       : {SEED}")

print()
print("PyTorch:")
print(f"  Version    : {torch.__version__}")
print(f"  CUDA       : {torch.cuda.is_available()}")

print()
print("Device:")
print(f"  {device}")

if torch.cuda.is_available():
    print(f"  GPU        : {torch.cuda.get_device_name(0)}")
    print(
        f"  CUDA       : "
        f"{torch.version.cuda}"
    )

print()
print("Output:")
print(f"  Root       : {ROOT_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Results    : {RESULTS_DIR}")
print(f"  Predictions: {PREDICTIONS_DIR}")
print(f"  Export     : {EXPORT_DIR}")

print()
print("✓ Dataset path validated.")
print("✓ Output directories created.")
print("✓ Reproducibility seed configured.")
print("✓ Strong-model configuration initialized.")

SIH26053 — 08 POINTNET++ STRONGER MODEL

Dataset:
  Path       : /kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1
  Exists     : True

Model:
  Name       : pointnet_pp_strong
  Input      : ['x', 'y', 'z', 'intensity', 'range']
  Channels   : 5
  Classes    : 4

Training:
  Epochs     : 50
  Batch size : 4
  LR         : 0.001
  Weight decay: 0.0001
  Seed       : 42

PyTorch:
  Version    : 2.10.0+cu128
  CUDA       : True

Device:
  cuda
  GPU        : Tesla T4
  CUDA       : 12.8

Output:
  Root       : /kaggle/working/strong_model
  Checkpoints: /kaggle/working/strong_model/checkpoints
  Results    : /kaggle/working/strong_model/results
  Predictions: /kaggle/working/strong_model/predictions
  Export     : /kaggle/working/strong_model/export

✓ Dataset path validated.
✓ Output directories created.
✓ Reproducibility seed configured.
✓ Strong-model configuration initialized.


In [2]:
# ============================================================================
# SIH26053 — STRONG MODEL METADATA VALIDATION
# ============================================================================

import json
from pathlib import Path

METADATA_DIR = DATASET_DIR / "metadata"

# ---------------------------------------------------------------------------
# Load metadata
# ---------------------------------------------------------------------------

with open(METADATA_DIR / "classes.json", "r") as f:
    classes_meta = json.load(f)

with open(METADATA_DIR / "preprocessing_config.json", "r") as f:
    preprocessing_config = json.load(f)

with open(METADATA_DIR / "train_manifest.json", "r") as f:
    train_manifest = json.load(f)

with open(METADATA_DIR / "val_manifest.json", "r") as f:
    val_manifest = json.load(f)

# ---------------------------------------------------------------------------
# Expected configuration
# ---------------------------------------------------------------------------

expected_class_to_id = {
    "drivable": 0,
    "non_drivable": 1,
    "static_obstacle": 2,
    "dynamic_object": 3,
}

expected_features = ["x", "y", "z", "intensity", "range"]

# classes.json is a metadata dictionary in this dataset
assert classes_meta["class_to_id"] == expected_class_to_id, (
    f"Unexpected class configuration:\n{classes_meta}"
)

assert classes_meta["ignore_class"]["id"] == 255
assert classes_meta["ignore_class"]["name"] == "unknown"
assert classes_meta["num_training_classes"] == 4

assert preprocessing_config["feature_names"] == expected_features
assert preprocessing_config["feature_dimension"] == 5

assert len(train_manifest) == 4
assert len(val_manifest) == 1

train_frames = sum(x["frame_count"] for x in train_manifest)
val_frames = sum(x["frame_count"] for x in val_manifest)

assert train_frames == 324
assert val_frames == 80
assert train_frames + val_frames == 404

# ---------------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------------

print("SIH26053 — STRONG MODEL METADATA")
print("=" * 75)

print("\nClasses:")
for name, idx in expected_class_to_id.items():
    print(f"  {idx}: {name}")

print("\nIgnore label:")
print("  255: unknown / ignore")

print("\nManifests:")
print(f"  Training shards   : {len(train_manifest)}")
print(f"  Validation shards : {len(val_manifest)}")
print(f"  Training frames   : {train_frames}")
print(f"  Validation frames : {val_frames}")
print(f"  Total frames      : {train_frames + val_frames}")

print("\nInput features:")
print(f"  Stored features   : {expected_features}")
print(f"  Feature dimension : {preprocessing_config['feature_dimension']}")
print("  Strong model input: ALL 5 features")

print("\nDataset:")
print(f"  Source            : {preprocessing_config.get('source_dataset', preprocessing_config.get('dataset', 'unknown'))}")
print(f"  Version           : {preprocessing_config.get('dataset_version', 'unknown')}")
print(f"  Max range         : {preprocessing_config.get('max_range_m', 'unknown')} m")

print("\n✓ Class configuration validated.")
print("✓ Ignore-label configuration validated.")
print("✓ Train/validation manifests validated.")
print("✓ Frame counts validated.")
print("✓ Five-feature configuration validated.")

print("\nCell 2 complete.")

SIH26053 — STRONG MODEL METADATA

Classes:
  0: drivable
  1: non_drivable
  2: static_obstacle
  3: dynamic_object

Ignore label:
  255: unknown / ignore

Manifests:
  Training shards   : 4
  Validation shards : 1
  Training frames   : 324
  Validation frames : 80
  Total frames      : 404

Input features:
  Stored features   : ['x', 'y', 'z', 'intensity', 'range']
  Feature dimension : 5
  Strong model input: ALL 5 features

Dataset:
  Source            : nuScenes-mini
  Version           : v1.0-mini
  Max range         : 100.0 m

✓ Class configuration validated.
✓ Ignore-label configuration validated.
✓ Train/validation manifests validated.
✓ Frame counts validated.
✓ Five-feature configuration validated.

Cell 2 complete.


In [4]:
# ============================================================================
# SIH26053 — STRONG POINTNET++ MODEL — CORRECTED
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Farthest Point Sampling
# ---------------------------------------------------------------------------

def farthest_point_sample(xyz, n_samples):
    """
    xyz:
        (B, N, 3)

    Returns:
        indices: (B, n_samples)
    """
    B, N, _ = xyz.shape

    n_samples = min(n_samples, N)

    centroids = torch.zeros(
        B, n_samples,
        dtype=torch.long,
        device=xyz.device
    )

    distance = torch.full(
        (B, N),
        float("inf"),
        device=xyz.device
    )

    farthest = torch.randint(
        0, N,
        (B,),
        dtype=torch.long,
        device=xyz.device
    )

    batch_idx = torch.arange(
        B,
        device=xyz.device
    )

    for i in range(n_samples):
        centroids[:, i] = farthest

        centroid_xyz = xyz[
            batch_idx,
            farthest
        ].unsqueeze(1)

        dist = torch.sum(
            (xyz - centroid_xyz) ** 2,
            dim=-1
        )

        distance = torch.minimum(
            distance,
            dist
        )

        farthest = torch.argmax(
            distance,
            dim=-1
        )

    return centroids


# ---------------------------------------------------------------------------
# Radius neighborhood grouping
# ---------------------------------------------------------------------------

def ball_query_group(
    xyz,
    new_xyz,
    radius,
    n_neighbors
):
    """
    xyz:
        (B, N, 3)

    new_xyz:
        (B, S, 3)

    Returns:
        indices:
        (B, S, K)
    """

    B, N, _ = xyz.shape

    dists = torch.cdist(
        new_xyz,
        xyz
    )

    idx = torch.topk(
        dists,
        k=min(n_neighbors, N),
        dim=-1,
        largest=False
    ).indices

    grouped_dists = torch.gather(
        dists,
        2,
        idx
    )

    within_radius = grouped_dists <= radius

    nearest = idx[:, :, :1].expand_as(idx)

    idx = torch.where(
        within_radius,
        idx,
        nearest
    )

    return idx


# ---------------------------------------------------------------------------
# Set Abstraction Layer
# ---------------------------------------------------------------------------

class StrongSetAbstraction(nn.Module):

    def __init__(
        self,
        n_samples,
        radius,
        n_neighbors,
        in_channels,
        mlp_channels
    ):
        super().__init__()

        self.n_samples = n_samples
        self.radius = radius
        self.n_neighbors = n_neighbors

        layers = []

        # Relative XYZ (3) + point features
        last_channels = in_channels + 3

        for out_channels in mlp_channels:
            layers.extend([
                nn.Conv2d(
                    last_channels,
                    out_channels,
                    kernel_size=1,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            ])

            last_channels = out_channels

        self.mlp = nn.Sequential(*layers)

    def forward(
        self,
        xyz,
        features
    ):
        B, N, _ = xyz.shape

        # ---------------------------------------------------------------
        # Farthest point sampling
        # ---------------------------------------------------------------

        centroid_idx = farthest_point_sample(
            xyz,
            self.n_samples
        )

        batch_idx = torch.arange(
            B,
            device=xyz.device
        ).view(B, 1)

        new_xyz = xyz[
            batch_idx,
            centroid_idx
        ]

        # ---------------------------------------------------------------
        # Neighborhood grouping
        # ---------------------------------------------------------------

        group_idx = ball_query_group(
            xyz=xyz,
            new_xyz=new_xyz,
            radius=self.radius,
            n_neighbors=self.n_neighbors
        )

        batch_idx_group = torch.arange(
            B,
            device=xyz.device
        ).view(B, 1, 1)

        grouped_xyz = xyz[
            batch_idx_group,
            group_idx
        ]

        # Relative coordinates
        grouped_xyz = (
            grouped_xyz
            - new_xyz.unsqueeze(2)
        )

        # ---------------------------------------------------------------
        # Add point features
        # ---------------------------------------------------------------

        if features is not None:

            grouped_features = features[
                batch_idx_group,
                group_idx
            ]

            grouped = torch.cat(
                [
                    grouped_xyz,
                    grouped_features
                ],
                dim=-1
            )

        else:
            grouped = grouped_xyz

        # ---------------------------------------------------------------
        # Conv2D expects:
        # (B, C, K, S)
        # ---------------------------------------------------------------

        grouped = grouped.permute(
            0, 3, 2, 1
        )

        new_features = self.mlp(
            grouped
        )

        # Local max pooling
        new_features = new_features.max(
            dim=2
        )[0]

        # (B, C, S) -> (B, S, C)
        new_features = new_features.permute(
            0, 2, 1
        )

        return new_xyz, new_features


# ---------------------------------------------------------------------------
# Feature Propagation
# ---------------------------------------------------------------------------

class FeaturePropagation(nn.Module):

    def __init__(
        self,
        in_channels,
        mlp_channels
    ):
        super().__init__()

        layers = []
        last = in_channels

        for out_channels in mlp_channels:

            layers.extend([
                nn.Conv1d(
                    last,
                    out_channels,
                    kernel_size=1,
                    bias=False
                ),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=True),
            ])

            last = out_channels

        self.mlp = nn.Sequential(*layers)

    def forward(
        self,
        xyz_target,
        xyz_source,
        features_target,
        features_source
    ):

        # ---------------------------------------------------------------
        # Nearest-neighbor interpolation
        # ---------------------------------------------------------------

        dists = torch.cdist(
            xyz_target,
            xyz_source
        )

        nearest_idx = dists.argmin(
            dim=-1
        )

        B = xyz_target.shape[0]

        batch_idx = torch.arange(
            B,
            device=xyz_target.device
        ).view(B, 1)

        interpolated = features_source[
            batch_idx,
            nearest_idx
        ]

        # ---------------------------------------------------------------
        # Skip connection
        # ---------------------------------------------------------------

        if features_target is not None:

            combined = torch.cat(
                [
                    interpolated,
                    features_target
                ],
                dim=-1
            )

        else:
            combined = interpolated

        # ---------------------------------------------------------------
        # MLP
        # ---------------------------------------------------------------

        combined = combined.permute(
            0, 2, 1
        )

        output = self.mlp(
            combined
        )

        output = output.permute(
            0, 2, 1
        )

        return output


# ============================================================================
# STRONG POINTNET++ SEGMENTER
# ============================================================================

class StrongPointNetPPSegmenter(nn.Module):

    """
    Stronger PointNet++-style point segmentation model.

    Input:
        (B, N, 5)
        [x, y, z, intensity, range]

    Output:
        (B, N, 4)
        class logits
    """

    def __init__(
        self,
        num_classes=4,
        in_channels=5
    ):
        super().__init__()

        # The input has 5 channels:
        # xyz = 3
        # intensity + range = 2
        #
        # Set-abstraction feature channels therefore equal 2.
        feature_channels = in_channels - 3

        assert feature_channels == 2, (
            f"Expected 5 total input features "
            f"(xyz + intensity + range), got {in_channels}"
        )

        # ---------------------------------------------------------------
        # Level 1
        # ---------------------------------------------------------------

        self.sa1 = StrongSetAbstraction(
            n_samples=1024,
            radius=1.0,
            n_neighbors=32,
            in_channels=feature_channels,
            mlp_channels=[
                64,
                64,
                128
            ]
        )

        # ---------------------------------------------------------------
        # Level 2
        # ---------------------------------------------------------------

        self.sa2 = StrongSetAbstraction(
            n_samples=256,
            radius=4.0,
            n_neighbors=64,
            in_channels=128,
            mlp_channels=[
                128,
                128,
                256
            ]
        )

        # ---------------------------------------------------------------
        # Feature propagation
        # ---------------------------------------------------------------

        self.fp1 = FeaturePropagation(
            in_channels=256 + 128,
            mlp_channels=[
                256,
                128
            ]
        )

        # Original-point skip connection contains:
        # intensity + range = 2 channels
        self.fp2 = FeaturePropagation(
            in_channels=128 + feature_channels,
            mlp_channels=[
                128,
                64
            ]
        )

        # ---------------------------------------------------------------
        # Classification head
        # ---------------------------------------------------------------

        self.head = nn.Sequential(
            nn.Conv1d(
                64,
                64,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),

            nn.Dropout(p=0.2),

            nn.Conv1d(
                64,
                num_classes,
                kernel_size=1
            )
        )

    def forward(self, points):

        # ---------------------------------------------------------------
        # Input:
        # (B, N, 5)
        # ---------------------------------------------------------------

        assert points.shape[-1] == 5, (
            f"Expected 5 input features, got {points.shape[-1]}"
        )

        xyz = points[..., :3]

        # Only non-coordinate features are passed as point features.
        point_features = points[..., 3:5]

        # ---------------------------------------------------------------
        # Hierarchical abstraction
        # ---------------------------------------------------------------

        xyz1, feat1 = self.sa1(
            xyz,
            point_features
        )

        xyz2, feat2 = self.sa2(
            xyz1,
            feat1
        )

        # ---------------------------------------------------------------
        # Propagate level 2 -> level 1
        # ---------------------------------------------------------------

        feat1_up = self.fp1(
            xyz_target=xyz1,
            xyz_source=xyz2,
            features_target=feat1,
            features_source=feat2
        )

        # ---------------------------------------------------------------
        # Propagate level 1 -> original points
        # ---------------------------------------------------------------

        feat0_up = self.fp2(
            xyz_target=xyz,
            xyz_source=xyz1,
            features_target=point_features,
            features_source=feat1_up
        )

        # ---------------------------------------------------------------
        # Classification
        # ---------------------------------------------------------------

        logits = self.head(
            feat0_up.permute(0, 2, 1)
        )

        return logits.permute(
            0,
            2,
            1
        )


# ============================================================================
# MODEL CREATION + SHAPE SMOKE TEST
# ============================================================================

strong_model = StrongPointNetPPSegmenter(
    num_classes=4,
    in_channels=5
).to(device)

print("SIH26053 — STRONG MODEL")
print("=" * 75)

print(strong_model)

parameter_count = sum(
    p.numel()
    for p in strong_model.parameters()
    if p.requires_grad
)

print(f"\nTrainable parameters: {parameter_count:,}")

# ---------------------------------------------------------------------------
# Shape smoke test
# ---------------------------------------------------------------------------

print("\nRunning shape smoke test...")

test_points = torch.randn(
    1,
    300,
    5,
    device=device
)

strong_model.eval()

with torch.no_grad():
    test_output = strong_model(
        test_points
    )

assert test_output.shape == (
    1,
    300,
    4
), test_output.shape

assert torch.isfinite(
    test_output
).all()

print(f"Input shape    : {tuple(test_points.shape)}")
print(f"Output shape   : {tuple(test_output.shape)}")
print("Expected shape : (1, 300, 4)")

print("\n✓ Strong PointNet++ model created.")
print("✓ Five input features accepted.")
print("✓ Four output classes produced.")
print("✓ Shape smoke test passed.")
print("✓ Output contains no NaN/Inf.")

print("\nCell 3 complete.")

SIH26053 — STRONG MODEL
StrongPointNetPPSegmenter(
  (sa1): StrongSetAbstraction(
    (mlp): Sequential(
      (0): Conv2d(5, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU(inplace=True)
    )
  )
  (sa2): StrongSetAbstraction(
    (mlp): Sequential(
      (0): Conv2d(131, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(1, 1), str

In [6]:
# ============================================================================
# SIH26053 — STRONG MODEL TRAINING DATASET
# ============================================================================

import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

print("=" * 75)
print("SIH26053 — STRONG MODEL TRAINING DATASET")
print("=" * 75)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/"
    "fovealidar_nuscenes_v1"
)

TRAIN_MANIFEST = DATASET_DIR / "metadata" / "train_manifest.json"

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"
assert TRAIN_MANIFEST.exists(), f"Training manifest not found: {TRAIN_MANIFEST}"

with open(TRAIN_MANIFEST, "r") as f:
    train_manifest = json.load(f)

print("\nTraining manifest:")
print(f"  Shards : {len(train_manifest)}")
print(f"  Frames : {sum(x['frame_count'] for x in train_manifest)}")

# ---------------------------------------------------------------------------
# Strong-model dataset
# ---------------------------------------------------------------------------

class StrongTrainingDataset(Dataset):

    def __init__(self, dataset_dir, manifest):
        self.dataset_dir = Path(dataset_dir)
        self.manifest = manifest

        self.frames = []

        for shard_info in manifest:
            shard_name = shard_info["shard"]
            frame_count = shard_info["frame_count"]

            for frame_idx in range(frame_count):
                self.frames.append(
                    (shard_name, frame_idx)
                )

        self._shard_cache = {}
        self.classes = [
            "drivable",
            "non_drivable",
            "static_obstacle",
            "dynamic_object",
        ]

        self.class_to_idx = {
            name: idx
            for idx, name in enumerate(self.classes)
        }

    def __len__(self):
        return len(self.frames)

    def _get_shard(self, shard_name):
        if shard_name not in self._shard_cache:
            path = self.dataset_dir / "train" / shard_name
            self._shard_cache[shard_name] = np.load(path)

        return self._shard_cache[shard_name]

    def __getitem__(self, index):

        shard_name, frame_idx = self.frames[index]

        data = self._get_shard(shard_name)

        points = data[f"points_{frame_idx}"].astype(np.float32)
        labels = data[f"labels_{frame_idx}"]

        # -------------------------------------------------------------------
        # Strong model uses ALL FIVE stored features:
        # x, y, z, intensity, range
        # -------------------------------------------------------------------

        assert points.ndim == 2
        assert points.shape[1] == 5

        points = torch.from_numpy(points).float()

        # Labels are stored as class names in the dataset.
        if labels.dtype.kind in {"U", "S", "O"}:
            labels = np.array(
                [self.class_to_idx[str(x)] for x in labels],
                dtype=np.int64,
            )
        else:
            labels = labels.astype(np.int64)

        # 255 = unknown / ignore
        labels[labels == 255] = -100

        labels = torch.from_numpy(labels).long()

        assert points.shape[0] == labels.shape[0]

        return {
            "points": points,
            "labels": labels,
        }


# ---------------------------------------------------------------------------
# Variable-size point clouds need padding
# ---------------------------------------------------------------------------

def strong_collate_fn(batch):

    batch_size = len(batch)
    max_points = max(item["points"].shape[0] for item in batch)

    points = torch.zeros(
        batch_size,
        max_points,
        5,
        dtype=torch.float32,
    )

    labels = torch.full(
        (batch_size, max_points),
        -100,
        dtype=torch.long,
    )

    for i, item in enumerate(batch):

        n = item["points"].shape[0]

        points[i, :n] = item["points"]
        labels[i, :n] = item["labels"]

    return {
        "points": points,
        "labels": labels,
    }


# ---------------------------------------------------------------------------
# Create dataset
# ---------------------------------------------------------------------------

strong_train_dataset = StrongTrainingDataset(
    DATASET_DIR,
    train_manifest,
)

print("\nDataset:")
print(f"  Training frames : {len(strong_train_dataset)}")

assert len(strong_train_dataset) == 324

# ---------------------------------------------------------------------------
# Create DataLoader
# ---------------------------------------------------------------------------

BATCH_SIZE = 4

train_loader = DataLoader(
    strong_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    collate_fn=strong_collate_fn,
)

print("\nDataLoader:")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Batches / epoch : {len(train_loader)}")

# ---------------------------------------------------------------------------
# Inspect one real batch
# ---------------------------------------------------------------------------

batch = next(iter(train_loader))

points = batch["points"]
labels = batch["labels"]

print("\nSample batch:")
print(f"  Points shape : {tuple(points.shape)}")
print(f"  Labels shape : {tuple(labels.shape)}")

print(
    f"  Points dtype : {points.dtype}"
)

print(
    f"  Labels found : {torch.unique(labels).tolist()}"
)

valid_points = (labels != -100).sum().item()
ignored_points = (labels == -100).sum().item()

print(f"  Valid points   : {valid_points:,}")
print(f"  Ignored points : {ignored_points:,}")

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

assert points.ndim == 3
assert points.shape[0] == BATCH_SIZE
assert points.shape[2] == 5

assert labels.ndim == 2
assert labels.shape[:2] == points.shape[:2]

allowed_labels = {-100, 0, 1, 2, 3}
found_labels = set(torch.unique(labels).tolist())

assert found_labels.issubset(allowed_labels), (
    f"Unexpected labels: {found_labels}"
)

assert valid_points > 0
assert ignored_points > 0

print("\n" + "=" * 75)
print("✓ STRONG TRAINING DATASET READY")
print("=" * 75)
print("✓ 324 training frames loaded.")
print("✓ Five stored features preserved.")
print("✓ Variable frame sizes handled by padding.")
print("✓ Ignore label 255 mapped to -100.")
print("✓ Four-class labels validated.")
print("✓ DataLoader created successfully.")
print("✓ Real training batch inspected.")

print("\nCell 4 complete.")

SIH26053 — STRONG MODEL TRAINING DATASET

Training manifest:
  Shards : 4
  Frames : 324

Dataset:
  Training frames : 324

DataLoader:
  Batch size      : 4
  Batches / epoch : 81

Sample batch:
  Points shape : (4, 34687, 5)
  Labels shape : (4, 34687)
  Points dtype : torch.float32
  Labels found : [-100, 0, 1, 2, 3]
  Valid points   : 103,157
  Ignored points : 35,591

✓ STRONG TRAINING DATASET READY
✓ 324 training frames loaded.
✓ Five stored features preserved.
✓ Variable frame sizes handled by padding.
✓ Ignore label 255 mapped to -100.
✓ Four-class labels validated.
✓ DataLoader created successfully.
✓ Real training batch inspected.

Cell 4 complete.


In [7]:
# ============================================================================
# SIH26053 — STRONG MODEL TRAINING BATCH SMOKE TEST
# ============================================================================

import torch

print("=" * 75)
print("SIH26053 — STRONG MODEL TRAINING BATCH SMOKE TEST")
print("=" * 75)

# ---------------------------------------------------------------------------
# Get a real training batch
# ---------------------------------------------------------------------------

batch = next(iter(train_loader))

points = batch["points"].to(
    device,
    non_blocking=True,
)

labels = batch["labels"].to(
    device,
    non_blocking=True,
)

print("\nBatch:")
print(f"  Points : {tuple(points.shape)}")
print(f"  Labels : {tuple(labels.shape)}")
print(f"  Device : {points.device}")

assert points.ndim == 3
assert points.shape[2] == 5

assert labels.ndim == 2
assert labels.shape[:2] == points.shape[:2]

# ---------------------------------------------------------------------------
# Forward pass
# ---------------------------------------------------------------------------

strong_model.train()

optimizer = torch.optim.Adam(
    strong_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

criterion = torch.nn.CrossEntropyLoss(
    ignore_index=-100
)

optimizer.zero_grad(set_to_none=True)

with torch.amp.autocast(
    device_type="cuda",
    enabled=(device.type == "cuda"),
):
    logits = strong_model(points)

    loss = criterion(
        logits.reshape(-1, 4),
        labels.reshape(-1),
    )

print("\nForward pass:")
print(f"  Logits shape : {tuple(logits.shape)}")
print(f"  Loss         : {loss.item():.4f}")

assert logits.shape == (
    points.shape[0],
    points.shape[1],
    4,
)

assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

# ---------------------------------------------------------------------------
# Backward pass
# ---------------------------------------------------------------------------

loss.backward()

parameters_with_grad = 0
finite_gradients = True

for parameter in strong_model.parameters():

    if parameter.grad is not None:

        parameters_with_grad += 1

        if not torch.isfinite(parameter.grad).all():
            finite_gradients = False

print("\nBackward pass:")
print(
    f"  Parameters with gradients : "
    f"{parameters_with_grad}"
)
print(
    f"  Gradients finite           : "
    f"{finite_gradients}"
)

assert parameters_with_grad > 0
assert finite_gradients

# ---------------------------------------------------------------------------
# Optimizer update
# ---------------------------------------------------------------------------

optimizer.step()

print("\nOptimizer:")
print("  Adam update completed.")

print("\n" + "=" * 75)
print("✓ STRONG MODEL TRAINING BATCH PASSED")
print("=" * 75)
print("✓ Real training batch forwarded successfully.")
print("✓ Five-feature input accepted.")
print("✓ Four-class logits produced.")
print("✓ CrossEntropyLoss computed.")
print("✓ Ignore padding (-100) handled.")
print("✓ Backpropagation completed.")
print("✓ Gradients are finite.")
print("✓ Optimizer update completed.")

print("\nCell 5 complete.")

SIH26053 — STRONG MODEL TRAINING BATCH SMOKE TEST

Batch:
  Points : (4, 34755, 5)
  Labels : (4, 34755)
  Device : cuda:0

Forward pass:
  Logits shape : (4, 34755, 4)
  Loss         : 1.4737

Backward pass:
  Parameters with gradients : 35
  Gradients finite           : True

Optimizer:
  Adam update completed.

✓ STRONG MODEL TRAINING BATCH PASSED
✓ Real training batch forwarded successfully.
✓ Five-feature input accepted.
✓ Four-class logits produced.
✓ CrossEntropyLoss computed.
✓ Ignore padding (-100) handled.
✓ Backpropagation completed.
✓ Gradients are finite.
✓ Optimizer update completed.

Cell 5 complete.


In [9]:
# ============================================================================
# SIH26053 — CALCULATING STRONG MODEL CLASS WEIGHTS
# ============================================================================

import torch
from collections import Counter

print("=" * 75)
print("SIH26053 — CALCULATING STRONG MODEL CLASS WEIGHTS")
print("=" * 75)

# ---------------------------------------------------------------------------
# Count labels from TRAINING DATA ONLY
# ---------------------------------------------------------------------------

class_counts = Counter()
ignore_count = 0

for batch_idx, batch in enumerate(train_loader):

    labels = batch["labels"]

    valid_mask = labels != -100

    valid_labels = labels[valid_mask]

    for c in range(4):
        class_counts[c] += (
            (valid_labels == c).sum().item()
        )

    ignore_count += (
        (labels == -100).sum().item()
    )

print("\nTraining class counts:")

class_names = [
    "drivable",
    "non_drivable",
    "static_obstacle",
    "dynamic_object",
]

for c, name in enumerate(class_names):
    print(
        f"  {c}: {name:18s} "
        f"{class_counts[c]:,}"
    )

print(
    f"\nTraining ignore points (255): "
    f"{ignore_count:,}"
)

# ---------------------------------------------------------------------------
# Validate counts
# ---------------------------------------------------------------------------

counts = torch.tensor(
    [class_counts[c] for c in range(4)],
    dtype=torch.float64,
)

assert torch.all(counts > 0), (
    f"Invalid class counts: {counts.tolist()}"
)

# ---------------------------------------------------------------------------
# Inverse-frequency class weighting
#
# weight_c = total_valid / (num_classes * count_c)
# ---------------------------------------------------------------------------

total_valid = counts.sum()

class_weights = total_valid / (
    len(class_names) * counts
)

# Normalize mean weight to exactly 1.0
class_weights = class_weights / class_weights.mean()

print("\nClass weights:")

for c, name in enumerate(class_names):
    print(
        f"  {c}: {name:18s} "
        f"{class_weights[c].item():.4f}"
    )

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

assert torch.isfinite(class_weights).all()
assert torch.all(class_weights > 0)

mean_weight = class_weights.mean().item()

print("\n" + "=" * 75)
print("STRONG MODEL CLASS WEIGHT SUMMARY")
print("=" * 75)

print(
    f"Valid training points : "
    f"{int(total_valid.item()):,}"
)

print(
    f"Classes               : "
    f"{len(class_names)}"
)

print(
    f"Mean weight           : "
    f"{mean_weight:.4f}"
)

print(
    "Ignore label          : "
    "255 / -100 (excluded)"
)

assert abs(mean_weight - 1.0) < 1e-6

# ---------------------------------------------------------------------------
# Move weights to CUDA
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Move weights to the actual model device
# ---------------------------------------------------------------------------

model_device = next(strong_model.parameters()).device

class_weights = class_weights.float().to(model_device)

print("\nCUDA class weights:")
for c, name in enumerate(class_names):
    print(
        f"  {c}: {name:18s} "
        f"{class_weights[c].item():.4f}"
    )

print(f"\nWeight device: {class_weights.device}")
print(f"Model device : {model_device}")

assert class_weights.device == model_device
assert torch.isfinite(class_weights).all()
assert torch.all(class_weights > 0)

print("\n" + "=" * 75)
print("✓ STRONG MODEL CLASS WEIGHTS READY")
print("=" * 75)
print("✓ Training-only class counts calculated.")
print("✓ Ignore labels excluded.")
print("✓ All four classes have valid counts.")
print("✓ Class weights are finite and positive.")
print("✓ Mean class weight normalized to 1.0.")
print("✓ Class weights placed on the model device.")

print("\nCell 6 complete.")

SIH26053 — CALCULATING STRONG MODEL CLASS WEIGHTS

Training class counts:
  0: drivable           3,088,835
  1: non_drivable       2,725,858
  2: static_obstacle    1,604,714
  3: dynamic_object     864,167

Training ignore points (255): 2,974,438

Class weights:
  0: drivable           0.5241
  1: non_drivable       0.5939
  2: static_obstacle    1.0088
  3: dynamic_object     1.8733

STRONG MODEL CLASS WEIGHT SUMMARY
Valid training points : 8,283,574
Classes               : 4
Mean weight           : 1.0000
Ignore label          : 255 / -100 (excluded)

CUDA class weights:
  0: drivable           0.5241
  1: non_drivable       0.5939
  2: static_obstacle    1.0088
  3: dynamic_object     1.8733

Weight device: cuda:0
Model device : cuda:0

✓ STRONG MODEL CLASS WEIGHTS READY
✓ Training-only class counts calculated.
✓ Ignore labels excluded.
✓ All four classes have valid counts.
✓ Class weights are finite and positive.
✓ Mean class weight normalized to 1.0.
✓ Class weights placed on th

In [11]:
# ============================================================================
# SIH26053 — STRONG MODEL WEIGHTED LOSS SMOKE TEST
# ============================================================================

import torch

print("=" * 75)
print("SIH26053 — STRONG MODEL WEIGHTED LOSS SMOKE TEST")
print("=" * 75)

# ---------------------------------------------------------------------------
# 1. Derive the actual model device
# ---------------------------------------------------------------------------

model_device = next(strong_model.parameters()).device

# Make the global device consistent with the model.
device = model_device

# Ensure class weights are on exactly the same device.
class_weights = class_weights.to(device)

# ---------------------------------------------------------------------------
# 2. Get a real training batch
# ---------------------------------------------------------------------------

batch = next(iter(train_loader))

points = batch["points"].to(device, non_blocking=True)
labels = batch["labels"].to(device, non_blocking=True)

valid_mask = labels != -100
valid_points = valid_mask.sum().item()
ignored_points = (~valid_mask).sum().item()

print("\nBatch:")
print(f"  Points        : {tuple(points.shape)}")
print(f"  Labels        : {tuple(labels.shape)}")
print(f"  Device        : {points.device}")
print(f"  Valid points  : {valid_points:,}")
print(f"  Ignored points: {ignored_points:,}")
print(f"  Model device  : {model_device}")
print(f"  Weight device : {class_weights.device}")

# ---------------------------------------------------------------------------
# 3. Device consistency checks
# ---------------------------------------------------------------------------

assert points.device == model_device, (
    f"Points on {points.device}, model on {model_device}"
)

assert labels.device == model_device, (
    f"Labels on {labels.device}, model on {model_device}"
)

assert class_weights.device == model_device, (
    f"Class weights on {class_weights.device}, model on {model_device}"
)

# ---------------------------------------------------------------------------
# 4. Weighted CrossEntropyLoss
# ---------------------------------------------------------------------------

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100,
)

strong_model.train()
strong_model.zero_grad(set_to_none=True)

# ---------------------------------------------------------------------------
# 5. Forward pass
# ---------------------------------------------------------------------------

with torch.amp.autocast(
    device_type="cuda" if model_device.type == "cuda" else "cpu",
    enabled=(model_device.type == "cuda"),
):
    logits = strong_model(points)

    loss = criterion(
        logits.reshape(-1, 4),
        labels.reshape(-1),
    )

print("\nForward pass:")
print(f"  Logits shape : {tuple(logits.shape)}")
print(f"  Weighted loss: {loss.item():.4f}")

# ---------------------------------------------------------------------------
# 6. Validate numerical stability
# ---------------------------------------------------------------------------

assert logits.shape == (
    points.shape[0],
    points.shape[1],
    4,
), f"Unexpected logits shape: {logits.shape}"

assert torch.isfinite(logits).all(), \
    "Logits contain NaN/Inf"

assert torch.isfinite(loss), \
    "Weighted loss is NaN/Inf"

# ---------------------------------------------------------------------------
# 7. Backward pass
# ---------------------------------------------------------------------------

loss.backward()

params_with_grad = 0
finite_gradients = True

for parameter in strong_model.parameters():
    if parameter.grad is not None:
        params_with_grad += 1

        if not torch.isfinite(parameter.grad).all():
            finite_gradients = False

print("\nBackward pass:")
print(f"  Parameters with gradients : {params_with_grad}")
print(f"  Gradients finite           : {finite_gradients}")

assert params_with_grad > 0
assert finite_gradients

# ---------------------------------------------------------------------------
# 8. Optimizer update
# ---------------------------------------------------------------------------

optimizer.step()

print("\nOptimizer:")
print("  Adam update completed.")

print("\n" + "=" * 75)
print("✓ STRONG MODEL WEIGHTED LOSS SMOKE TEST PASSED")
print("=" * 75)

print("✓ Real training batch forwarded successfully.")
print("✓ Five-feature input accepted.")
print("✓ Four-class logits produced.")
print("✓ Class-weighted CrossEntropyLoss computed.")
print("✓ Ignore padding (-100) handled correctly.")
print("✓ Loss is finite.")
print("✓ Backpropagation completed.")
print("✓ Gradients are finite.")
print("✓ Optimizer update completed.")
print("✓ Model, batch, labels, and class weights use one device.")

print("\nCell 7 complete.")

SIH26053 — STRONG MODEL WEIGHTED LOSS SMOKE TEST

Batch:
  Points        : (4, 34719, 5)
  Labels        : (4, 34719)
  Device        : cuda:0
  Valid points  : 102,976
  Ignored points: 35,900
  Model device  : cuda:0
  Weight device : cuda:0

Forward pass:
  Logits shape : (4, 34719, 4)
  Weighted loss: 1.4328

Backward pass:
  Parameters with gradients : 35
  Gradients finite           : True

Optimizer:
  Adam update completed.

✓ STRONG MODEL WEIGHTED LOSS SMOKE TEST PASSED
✓ Real training batch forwarded successfully.
✓ Five-feature input accepted.
✓ Four-class logits produced.
✓ Class-weighted CrossEntropyLoss computed.
✓ Ignore padding (-100) handled correctly.
✓ Loss is finite.
✓ Backpropagation completed.
✓ Gradients are finite.
✓ Optimizer update completed.
✓ Model, batch, labels, and class weights use one device.

Cell 7 complete.


In [13]:
# ===========================================================================
# SIH26053 — 08 STRONG MODEL FULL TRAINING
# ===========================================================================

import torch
import json
import time
from pathlib import Path

print("=" * 75)
print("SIH26053 — STRONG MODEL FULL TRAINING")
print("=" * 75)

# ===========================================================================
# 1. CONFIGURATION
# ===========================================================================

EPOCHS = 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
SEED = 42

# IMPORTANT:
# Define classes explicitly so checkpoint creation never depends on
# a missing notebook variable.
classes = [
    "drivable",
    "non_drivable",
    "static_obstacle",
    "dynamic_object",
]

NUM_CLASSES = len(classes)

INPUT_FEATURES = [
    "x",
    "y",
    "z",
    "intensity",
    "range",
]

INPUT_DIM = len(INPUT_FEATURES)

IGNORE_INDEX = -100

# ===========================================================================
# 2. OUTPUT DIRECTORIES
# ===========================================================================

STRONG_ROOT = Path("/kaggle/working/strong_model")

CHECKPOINT_DIR = STRONG_ROOT / "checkpoints"
RESULTS_DIR = STRONG_ROOT / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = CHECKPOINT_DIR / "best_model.pth"
LOSS_HISTORY_PATH = RESULTS_DIR / "training_loss_history.json"

# ===========================================================================
# 3. REPRODUCIBILITY
# ===========================================================================

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===========================================================================
# 4. REQUIRED OBJECT VALIDATION
# ===========================================================================

assert "strong_model" in globals(), (
    "strong_model is not defined. "
    "Run Cell 3 before Cell 8."
)

assert "train_loader" in globals(), (
    "train_loader is not defined. "
    "Run Cell 4 before Cell 8."
)

assert "class_weights" in globals(), (
    "class_weights is not defined. "
    "Run Cell 6 before Cell 8."
)

assert "device" in globals(), (
    "device is not defined. "
    "Run Cell 1 before Cell 8."
)

# ===========================================================================
# 5. MODEL + CLASS WEIGHTS
# ===========================================================================

model = strong_model.to(device)

class_weights = class_weights.to(device)

model_device = next(model.parameters()).device

assert model_device == device, (
    f"Model device mismatch: "
    f"{model_device} != {device}"
)

assert class_weights.device == device, (
    f"Class-weight device mismatch: "
    f"{class_weights.device} != {device}"
)

# ===========================================================================
# 6. MODEL PARAMETER COUNT
# ===========================================================================

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nConfiguration:")
print(f"  Model          : StrongPointNetPPSegmenter")
print(f"  Epochs         : {EPOCHS}")
print(f"  Batch size     : {train_loader.batch_size}")
print(f"  Batches/epoch  : {len(train_loader)}")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Weight decay   : {WEIGHT_DECAY}")
print(f"  Device         : {device}")
print(f"  Parameters     : {parameter_count:,}")

print("\nInput features:")
for i, feature in enumerate(INPUT_FEATURES):
    print(f"  {i}: {feature}")

print("\nClasses:")
for i, cls in enumerate(classes):
    print(f"  {i}: {cls}")

print("\nClass weights:")
for i, weight in enumerate(class_weights.detach().cpu().tolist()):
    print(f"  {i}: {classes[i]:18s} {weight:.4f}")

# ===========================================================================
# 7. LOSS FUNCTION
# ===========================================================================

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX,
)

# ===========================================================================
# 8. OPTIMIZER
# ===========================================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# ===========================================================================
# 9. MIXED PRECISION
# ===========================================================================

use_amp = device.type == "cuda"

if use_amp:
    scaler = torch.amp.GradScaler("cuda")
else:
    scaler = None

print("\nMixed precision:")
print(f"  AMP enabled : {use_amp}")

# ===========================================================================
# 10. TRAINING STATE
# ===========================================================================

epoch_losses = []

best_loss = float("inf")
best_epoch = 0

training_start = time.perf_counter()

print("\n" + "=" * 75)
print("STARTING STRONG MODEL TRAINING")
print("=" * 75)

# ===========================================================================
# 11. FULL TRAINING LOOP
# ===========================================================================

for epoch in range(1, EPOCHS + 1):

    model.train()

    running_loss = 0.0
    batches_completed = 0

    epoch_start = time.perf_counter()

    for batch in train_loader:

        # ---------------------------------------------------------------
        # Load batch
        # ---------------------------------------------------------------

        points = batch["points"].to(
            device,
            non_blocking=True,
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True,
        )

        # ---------------------------------------------------------------
        # Basic batch validation
        # ---------------------------------------------------------------

        assert points.ndim == 3, (
            f"Expected points [B,N,C], got {points.shape}"
        )

        assert labels.ndim == 2, (
            f"Expected labels [B,N], got {labels.shape}"
        )

        assert points.shape[0] == labels.shape[0], (
            "Batch dimension mismatch."
        )

        assert points.shape[1] == labels.shape[1], (
            "Point/label count mismatch."
        )

        assert points.shape[2] == INPUT_DIM, (
            f"Expected {INPUT_DIM} input features, "
            f"got {points.shape[2]}"
        )

        # ---------------------------------------------------------------
        # Optimizer reset
        # ---------------------------------------------------------------

        optimizer.zero_grad(set_to_none=True)

        # ---------------------------------------------------------------
        # Forward + loss
        # ---------------------------------------------------------------

        if use_amp:

            with torch.amp.autocast("cuda"):

                logits = model(points)

                assert logits.shape[0] == points.shape[0]
                assert logits.shape[1] == points.shape[1]
                assert logits.shape[2] == NUM_CLASSES

                loss = criterion(
                    logits.reshape(-1, NUM_CLASSES),
                    labels.reshape(-1),
                )

            # -----------------------------------------------------------
            # Backward
            # -----------------------------------------------------------

            scaler.scale(loss).backward()

            # Unscale before gradient clipping.
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0,
            )

            scaler.step(optimizer)
            scaler.update()

        else:

            logits = model(points)

            assert logits.shape[0] == points.shape[0]
            assert logits.shape[1] == points.shape[1]
            assert logits.shape[2] == NUM_CLASSES

            loss = criterion(
                logits.reshape(-1, NUM_CLASSES),
                labels.reshape(-1),
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0,
            )

            optimizer.step()

        # ---------------------------------------------------------------
        # Loss validation
        # ---------------------------------------------------------------

        loss_value = float(loss.detach().item())

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss detected at "
                f"epoch {epoch}: {loss_value}"
            )

        running_loss += loss_value
        batches_completed += 1

    # =========================================================================
    # 12. EPOCH SUMMARY
    # =========================================================================

    epoch_loss = running_loss / batches_completed

    epoch_losses.append(epoch_loss)

    epoch_time = time.perf_counter() - epoch_start

    print(
        f"epoch {epoch:02d}/{EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"time={epoch_time:.1f}s"
    )

    # =========================================================================
    # 13. SAVE BEST CHECKPOINT
    # =========================================================================

    if epoch_loss < best_loss:

        best_loss = epoch_loss
        best_epoch = epoch

        checkpoint = {
            # -----------------------------------------------------------
            # Model identity
            # -----------------------------------------------------------

            "model_choice":
                "pointnet_pp_strong",

            # -----------------------------------------------------------
            # Weights
            # -----------------------------------------------------------

            "model_state_dict":
                model.state_dict(),

            # -----------------------------------------------------------
            # Training state
            # -----------------------------------------------------------

            "epoch":
                epoch,

            "loss":
                epoch_loss,

            "seed":
                SEED,

            # -----------------------------------------------------------
            # Classes
            # -----------------------------------------------------------

            "classes":
                classes,

            "num_classes":
                NUM_CLASSES,

            # -----------------------------------------------------------
            # Input configuration
            # -----------------------------------------------------------

            "input_features":
                INPUT_FEATURES,

            "input_dim":
                INPUT_DIM,

            # -----------------------------------------------------------
            # Architecture information
            # -----------------------------------------------------------

            "parameters":
                parameter_count,

            # -----------------------------------------------------------
            # Training configuration
            # -----------------------------------------------------------

            "training_config": {
                "epochs":
                    EPOCHS,

                "batch_size":
                    train_loader.batch_size,

                "learning_rate":
                    LEARNING_RATE,

                "weight_decay":
                    WEIGHT_DECAY,

                "ignore_index":
                    IGNORE_INDEX,

                "class_weights":
                    class_weights
                    .detach()
                    .cpu()
                    .tolist(),
            },
        }

        torch.save(
            checkpoint,
            BEST_CHECKPOINT,
        )

        print(
            f"  ✓ best checkpoint saved "
            f"(loss={epoch_loss:.4f})"
        )

# ===========================================================================
# 14. TRAINING TIME
# ===========================================================================

training_time = (
    time.perf_counter() - training_start
)

# ===========================================================================
# 15. SAVE COMPLETE LOSS HISTORY
# ===========================================================================

history = {
    "model":
        "pointnet_pp_strong",

    "epochs_requested":
        EPOCHS,

    "epochs_completed":
        len(epoch_losses),

    "best_epoch":
        best_epoch,

    "best_loss":
        best_loss,

    "final_loss":
        epoch_losses[-1],

    "epoch_losses":
        epoch_losses,

    "training_seconds":
        training_time,

    "training_minutes":
        training_time / 60.0,

    "training_config": {
        "batch_size":
            train_loader.batch_size,

        "batches_per_epoch":
            len(train_loader),

        "learning_rate":
            LEARNING_RATE,

        "weight_decay":
            WEIGHT_DECAY,

        "seed":
            SEED,

        "device":
            str(device),

        "gpu":
            (
                torch.cuda.get_device_name(0)
                if device.type == "cuda"
                else None
            ),
    },

    "classes":
        classes,

    "input_features":
        INPUT_FEATURES,

    "class_weights":
        class_weights
        .detach()
        .cpu()
        .tolist(),
}

with open(
    LOSS_HISTORY_PATH,
    "w",
) as f:

    json.dump(
        history,
        f,
        indent=2,
    )

# ===========================================================================
# 16. FINAL TRAINING VALIDATION
# ===========================================================================

assert len(epoch_losses) == EPOCHS, (
    f"Expected {EPOCHS} epochs, "
    f"got {len(epoch_losses)}"
)

assert all(
    torch.isfinite(
        torch.tensor(loss)
    )
    for loss in epoch_losses
)

assert BEST_CHECKPOINT.exists(), (
    "Best checkpoint was not created."
)

assert LOSS_HISTORY_PATH.exists(), (
    "Training history was not created."
)

# ===========================================================================
# 17. RELOAD BEST CHECKPOINT
# ===========================================================================

saved_checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
)

assert saved_checkpoint["model_choice"] == \
    "pointnet_pp_strong"

assert saved_checkpoint["classes"] == \
    classes

assert saved_checkpoint["num_classes"] == \
    NUM_CLASSES

assert saved_checkpoint["input_features"] == \
    INPUT_FEATURES

assert saved_checkpoint["input_dim"] == \
    INPUT_DIM

assert saved_checkpoint["parameters"] == \
    parameter_count

assert torch.isfinite(
    torch.tensor(
        saved_checkpoint["loss"]
    )
)

# ===========================================================================
# 18. FINAL REPORT
# ===========================================================================

print("\n" + "=" * 75)
print("STRONG MODEL TRAINING SUMMARY")
print("=" * 75)

print(f"Epochs completed : {len(epoch_losses)}")
print(f"Best epoch       : {best_epoch}")
print(f"Best loss        : {best_loss:.6f}")
print(f"Final loss       : {epoch_losses[-1]:.6f}")

print(
    f"Training time    : "
    f"{training_time / 60.0:.2f} minutes"
)

print("\nCheckpoint:")
print(f"  {BEST_CHECKPOINT}")

print("\nTraining history:")
print(f"  {LOSS_HISTORY_PATH}")

print("\nCheckpoint metadata:")
print(f"  Model          : {saved_checkpoint['model_choice']}")
print(f"  Epoch          : {saved_checkpoint['epoch']}")
print(f"  Loss           : {saved_checkpoint['loss']:.6f}")
print(f"  Parameters     : {saved_checkpoint['parameters']:,}")
print(f"  Input features : {saved_checkpoint['input_features']}")

print("\n" + "=" * 75)
print("✓ STRONG MODEL FULL TRAINING COMPLETE")
print("=" * 75)

print("✓ All 50 epochs completed.")
print("✓ All epoch losses are finite.")
print("✓ Best checkpoint saved.")
print("✓ Best checkpoint successfully reloaded.")
print("✓ Checkpoint metadata validated.")
print("✓ Training loss history saved.")
print("✓ Five-feature training completed.")
print("✓ Class-weighted CrossEntropyLoss used.")
print("✓ Ignore padding (-100) handled correctly.")

print("\nCell 8 complete.")

SIH26053 — STRONG MODEL FULL TRAINING

Configuration:
  Model          : StrongPointNetPPSegmenter
  Epochs         : 50
  Batch size     : 4
  Batches/epoch  : 81
  Learning rate  : 0.001
  Weight decay   : 0.0001
  Device         : cuda:0
  Parameters     : 241,604

Input features:
  0: x
  1: y
  2: z
  3: intensity
  4: range

Classes:
  0: drivable
  1: non_drivable
  2: static_obstacle
  3: dynamic_object

Class weights:
  0: drivable           0.5241
  1: non_drivable       0.5939
  2: static_obstacle    1.0088
  3: dynamic_object     1.8733

Mixed precision:
  AMP enabled : True

STARTING STRONG MODEL TRAINING
epoch 01/50 | loss=0.8097 | time=18.8s
  ✓ best checkpoint saved (loss=0.8097)
epoch 02/50 | loss=0.7484 | time=18.8s
  ✓ best checkpoint saved (loss=0.7484)
epoch 03/50 | loss=0.6845 | time=18.8s
  ✓ best checkpoint saved (loss=0.6845)
epoch 04/50 | loss=0.6479 | time=18.8s
  ✓ best checkpoint saved (loss=0.6479)
epoch 05/50 | loss=0.6624 | time=18.9s
epoch 06/50 | loss=

In [15]:
# ===========================================================================
# SIH26053 — 09 VALIDATE STRONG MODEL
# ===========================================================================

import torch
import json
import numpy as np
from pathlib import Path

print("=" * 75)
print("SIH26053 — 09 VALIDATE STRONG MODEL")
print("=" * 75)

# ===========================================================================
# PATHS
# ===========================================================================

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1"
)

STRONG_ROOT = Path(
    "/kaggle/working/strong_model"
)

CHECKPOINT_PATH = (
    STRONG_ROOT
    / "checkpoints"
    / "best_model.pth"
)

RESULTS_DIR = (
    STRONG_ROOT
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATION_JSON = (
    RESULTS_DIR
    / "validation_results.json"
)

VALIDATION_REPORT = (
    RESULTS_DIR
    / "STRONG_VALIDATION_REPORT.txt"
)

# ===========================================================================
# DEVICE
# ===========================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDataset:")
print(f"  Path   : {DATASET_DIR}")
print(f"  Exists : {DATASET_DIR.exists()}")

print("\nCheckpoint:")
print(f"  Path   : {CHECKPOINT_PATH}")
print(f"  Exists : {CHECKPOINT_PATH.exists()}")

print("\nDevice:")
print(f"  {device}")

if device.type == "cuda":
    print(
        f"  GPU : {torch.cuda.get_device_name(0)}"
    )

assert DATASET_DIR.exists(), (
    f"Dataset directory not found: {DATASET_DIR}"
)

assert CHECKPOINT_PATH.exists(), (
    f"Strong-model checkpoint not found: {CHECKPOINT_PATH}"
)

print("\n✓ Dataset path validated.")
print("✓ Strong-model checkpoint validated.")
print("\nContinue with the remainder of the Cell 9 validation code.")

SIH26053 — 09 VALIDATE STRONG MODEL

Dataset:
  Path   : /kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1
  Exists : True

Checkpoint:
  Path   : /kaggle/working/strong_model/checkpoints/best_model.pth
  Exists : True

Device:
  cuda
  GPU : Tesla T4

✓ Dataset path validated.
✓ Strong-model checkpoint validated.

Continue with the remainder of the Cell 9 validation code.


In [18]:
import inspect

print("StrongPointNetPPSegmenter constructor:")
print(inspect.signature(StrongPointNetPPSegmenter.__init__))

StrongPointNetPPSegmenter constructor:
(self, num_classes=4, in_channels=5)


In [19]:
print(type(strong_model))
print(strong_model)

<class '__main__.StrongPointNetPPSegmenter'>
StrongPointNetPPSegmenter(
  (sa1): StrongSetAbstraction(
    (mlp): Sequential(
      (0): Conv2d(5, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU(inplace=True)
    )
  )
  (sa2): StrongSetAbstraction(
    (mlp): Sequential(
      (0): Conv2d(131, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, ke

In [20]:
# ============================================================================
# SIH26053 — 09 VALIDATE STRONG MODEL
# ============================================================================
# Full validation of the trained PointNet++-style strong model.
#
# Uses:
#   - ALL 5 stored features: x, y, z, intensity, range
#   - 4 semantic classes
#   - ignore label 255
#   - best checkpoint from Cell 8
#
# Outputs:
#   /kaggle/working/strong_model/results/validation_results/
#       validation_results.json
#       VALIDATION_REPORT.txt
# ============================================================================

import torch
import torch.nn as nn
import numpy as np
import json
import time
from pathlib import Path
from collections import defaultdict

# ---------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/"
    "fovealidar-nuscenes-v1/fovealidar_nuscenes_v1"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/strong_model/checkpoints/best_model.pth"
)

RESULTS_DIR = Path(
    "/kaggle/working/strong_model/results/validation_results"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 75)
print("SIH26053 — 09 VALIDATE STRONG MODEL")
print("=" * 75)

print("\nDataset:")
print(f"  Path   : {DATASET_DIR}")
print(f"  Exists : {DATASET_DIR.exists()}")

print("\nCheckpoint:")
print(f"  Path   : {CHECKPOINT_PATH}")
print(f"  Exists : {CHECKPOINT_PATH.exists()}")

print("\nDevice:")
print(f"  {device}")
if device.type == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

# ---------------------------------------------------------------------------
# 2. LOAD CHECKPOINT METADATA
# ---------------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

required_keys = [
    "model_choice",
    "epoch",
    "loss",
    "seed",
    "classes",
]

for key in required_keys:
    assert key in checkpoint, f"Checkpoint missing key: {key}"

classes = checkpoint["classes"]

assert classes == [
    "drivable",
    "non_drivable",
    "static_obstacle",
    "dynamic_object",
]

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(classes)
}

NUM_CLASSES = len(classes)

print("\nCheckpoint metadata:")
print(f"  Model          : {checkpoint['model_choice']}")
print(f"  Best epoch     : {checkpoint['epoch']}")
print(f"  Training loss  : {checkpoint['loss']:.6f}")
print(f"  Seed           : {checkpoint['seed']}")

if "parameters" in checkpoint:
    print(f"  Parameters     : {checkpoint['parameters']:,}")

print("\nClasses:")
for i, cls in enumerate(classes):
    print(f"  {i}: {cls}")

assert checkpoint["model_choice"] == "pointnet_pp_strong"
assert NUM_CLASSES == 4

# ---------------------------------------------------------------------------
# 3. RECREATE EXACT STRONG MODEL
# ---------------------------------------------------------------------------
# IMPORTANT:
# Your actual constructor is:
#
#   StrongPointNetPPSegmenter(num_classes=4, in_channels=5)
#
# Do NOT use input_dim=5.
# ---------------------------------------------------------------------------

strong_model = StrongPointNetPPSegmenter(
    num_classes=NUM_CLASSES,
    in_channels=5,
)

strong_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

strong_model = strong_model.to(device)
strong_model.eval()

parameter_count = sum(
    p.numel()
    for p in strong_model.parameters()
    if p.requires_grad
)

print("\nModel:")
print(f"  Architecture : {strong_model.__class__.__name__}")
print(f"  Parameters   : {parameter_count:,}")
print(f"  Input        : 5 features")
print(f"  Output       : {NUM_CLASSES} classes")

assert parameter_count == 241_604, (
    f"Unexpected parameter count: {parameter_count}"
)

# ---------------------------------------------------------------------------
# 4. VALIDATE MODEL SHAPE
# ---------------------------------------------------------------------------

print("\nShape validation:")

test_points = torch.randn(
    1,
    300,
    5,
    device=device,
)

with torch.no_grad():
    test_output = strong_model(test_points)

print(f"  Input shape  : {tuple(test_points.shape)}")
print(f"  Output shape : {tuple(test_output.shape)}")

assert test_output.shape == (1, 300, NUM_CLASSES)
assert torch.isfinite(test_output).all()

del test_points
del test_output

print("  ✓ Five-feature input accepted.")
print("  ✓ Four-class output produced.")
print("  ✓ Output contains no NaN/Inf.")

# ---------------------------------------------------------------------------
# 5. LOAD VALIDATION MANIFEST
# ---------------------------------------------------------------------------

VAL_MANIFEST_PATH = DATASET_DIR / "metadata" / "val_manifest.json"

assert VAL_MANIFEST_PATH.exists(), (
    f"Validation manifest not found: {VAL_MANIFEST_PATH}"
)

with open(VAL_MANIFEST_PATH, "r") as f:
    val_manifest = json.load(f)

print("\nValidation manifest:")
print(f"  Shards : {len(val_manifest)}")

expected_frames = sum(
    int(item["frame_count"])
    for item in val_manifest
)

print(f"  Frames : {expected_frames}")

assert expected_frames == 80

# ---------------------------------------------------------------------------
# 6. IOU FUNCTION
# ---------------------------------------------------------------------------

IGNORE_INDEX = 255

def compute_intersection_union(
    preds,
    targets,
    num_classes,
):
    """
    Computes per-class intersection and union while excluding
    ignored target points.
    """

    valid = targets != IGNORE_INDEX

    preds = preds[valid]
    targets = targets[valid]

    intersections = np.zeros(
        num_classes,
        dtype=np.int64
    )

    unions = np.zeros(
        num_classes,
        dtype=np.int64
    )

    for c in range(num_classes):

        pred_c = preds == c
        target_c = targets == c

        intersections[c] = np.logical_and(
            pred_c,
            target_c
        ).sum()

        unions[c] = np.logical_or(
            pred_c,
            target_c
        ).sum()

    return intersections, unions


# ---------------------------------------------------------------------------
# 7. DISTANCE BANDS
# ---------------------------------------------------------------------------

BANDS = [
    (0, 10),
    (10, 30),
    (30, 60),
    (60, 100),
]

band_intersections = {
    f"{lo}-{hi}m": np.zeros(
        NUM_CLASSES,
        dtype=np.int64
    )
    for lo, hi in BANDS
}

band_unions = {
    f"{lo}-{hi}m": np.zeros(
        NUM_CLASSES,
        dtype=np.int64
    )
    for lo, hi in BANDS
}

# ---------------------------------------------------------------------------
# 8. GLOBAL ACCUMULATORS
# ---------------------------------------------------------------------------

global_intersections = np.zeros(
    NUM_CLASSES,
    dtype=np.int64
)

global_unions = np.zeros(
    NUM_CLASSES,
    dtype=np.int64
)

total_points = 0
valid_points = 0
ignored_points = 0
frames_evaluated = 0

# ---------------------------------------------------------------------------
# 9. FULL VALIDATION
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("FULL STRONG-MODEL VALIDATION")
print("=" * 75)

validation_start = time.perf_counter()

with torch.no_grad():

    for shard_index, shard_info in enumerate(val_manifest):

        shard_name = shard_info["shard"]

        shard_path = (
            DATASET_DIR
            / "val"
            / shard_name
        )

        assert shard_path.exists(), (
            f"Validation shard not found: {shard_path}"
        )

        data = np.load(
            shard_path,
            allow_pickle=False
        )

        frame_count = int(
            shard_info["frame_count"]
        )

        for i in range(frame_count):

            points_np = data[f"points_{i}"]
            labels_np = data[f"labels_{i}"]

            assert points_np.ndim == 2
            assert points_np.shape[1] == 5
            assert labels_np.ndim == 1
            assert points_np.shape[0] == labels_np.shape[0]

            total_points += points_np.shape[0]

            ignored_mask = (
                labels_np == IGNORE_INDEX
            )

            ignored_points += int(
                ignored_mask.sum()
            )

            valid_points += int(
                (~ignored_mask).sum()
            )

            # ---------------------------------------------------------------
            # STRONG MODEL USES ALL FIVE FEATURES
            # ---------------------------------------------------------------

            points = torch.from_numpy(
                points_np
            ).float().unsqueeze(0).to(device)

            logits = strong_model(points)

            assert logits.shape == (
                1,
                points_np.shape[0],
                NUM_CLASSES
            )

            assert torch.isfinite(logits).all()

            preds = (
                logits
                .argmax(dim=-1)
                .squeeze(0)
                .cpu()
                .numpy()
                .astype(np.int64)
            )

            # ---------------------------------------------------------------
            # GLOBAL METRICS
            # ---------------------------------------------------------------

            inter, union = compute_intersection_union(
                preds,
                labels_np,
                NUM_CLASSES,
            )

            global_intersections += inter
            global_unions += union

            # ---------------------------------------------------------------
            # DISTANCE
            # ---------------------------------------------------------------

            ranges = np.sqrt(
                points_np[:, 0] ** 2
                + points_np[:, 1] ** 2
            )

            for lo, hi in BANDS:

                band_name = f"{lo}-{hi}m"

                mask = (
                    (ranges >= lo)
                    & (ranges < hi)
                )

                if not np.any(mask):
                    continue

                band_inter, band_union = (
                    compute_intersection_union(
                        preds[mask],
                        labels_np[mask],
                        NUM_CLASSES,
                    )
                )

                band_intersections[
                    band_name
                ] += band_inter

                band_unions[
                    band_name
                ] += band_union

            frames_evaluated += 1

            if frames_evaluated % 10 == 0:
                print(
                    f"  evaluated "
                    f"{frames_evaluated}/{expected_frames} frames"
                )

        data.close()

validation_time = (
    time.perf_counter()
    - validation_start
)

# ---------------------------------------------------------------------------
# 10. CALCULATE GLOBAL IOU
# ---------------------------------------------------------------------------

global_ious = {}

for c, cls in enumerate(classes):

    if global_unions[c] > 0:
        global_ious[cls] = (
            float(global_intersections[c])
            / float(global_unions[c])
        )
    else:
        global_ious[cls] = None

valid_global_ious = [
    v
    for v in global_ious.values()
    if v is not None
]

global_miou = (
    float(np.mean(valid_global_ious))
    if valid_global_ious
    else None
)

# ---------------------------------------------------------------------------
# 11. DISTANCE-BAND IOU
# ---------------------------------------------------------------------------

band_results = {}

for lo, hi in BANDS:

    band_name = f"{lo}-{hi}m"

    class_results = {}

    for c, cls in enumerate(classes):

        union = band_unions[
            band_name
        ][c]

        if union > 0:
            class_results[cls] = (
                float(
                    band_intersections[
                        band_name
                    ][c]
                )
                / float(union)
            )
        else:
            class_results[cls] = None

    valid_values = [
        v
        for v in class_results.values()
        if v is not None
    ]

    band_miou = (
        float(np.mean(valid_values))
        if valid_values
        else None
    )

    band_results[band_name] = {
        "per_class_iou": class_results,
        "miou": band_miou,
    }

# ---------------------------------------------------------------------------
# 12. PRINT RESULTS
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("GLOBAL STRONG MODEL VALIDATION RESULTS")
print("=" * 75)

print(f"Frames evaluated : {frames_evaluated}")
print(f"Total points     : {total_points:,}")
print(f"Valid points     : {valid_points:,}")
print(f"Ignored points   : {ignored_points:,}")

print("\nPer-class IoU:")

for cls in classes:

    value = global_ious[cls]

    if value is None:
        print(f"  {cls:18s} IoU=None")
    else:
        print(
            f"  {cls:18s} IoU={value:.4f}"
        )

print(
    f"\nOverall mIoU       : {global_miou:.4f}"
)

print("\n" + "=" * 75)
print("DISTANCE-BAND STRONG MODEL RESULTS")
print("=" * 75)

for band_name, result in band_results.items():

    print(f"\n{band_name}:")

    for cls in classes:

        value = result["per_class_iou"][cls]

        if value is None:
            print(
                f"  {cls:18s} IoU=None"
            )
        else:
            print(
                f"  {cls:18s} IoU={value:.4f}"
            )

    print(
        f"  mIoU               = "
        f"{result['miou']:.4f}"
    )

# ---------------------------------------------------------------------------
# 13. SAVE JSON RESULTS
# ---------------------------------------------------------------------------

results = {
    "experiment": "SIH26053",
    "model": {
        "model_choice": checkpoint["model_choice"],
        "architecture": "StrongPointNetPPSegmenter",
        "parameters": parameter_count,
        "input_features": [
            "x",
            "y",
            "z",
            "intensity",
            "range",
        ],
        "num_classes": NUM_CLASSES,
        "classes": classes,
        "checkpoint_epoch": int(
            checkpoint["epoch"]
        ),
        "training_loss": float(
            checkpoint["loss"]
        ),
        "seed": int(
            checkpoint["seed"]
        ),
    },
    "dataset": {
        "source": "nuScenes-mini",
        "validation_shards": len(
            val_manifest
        ),
        "validation_frames": expected_frames,
        "frames_evaluated": frames_evaluated,
        "total_points": int(total_points),
        "valid_points": int(valid_points),
        "ignored_points": int(ignored_points),
        "ignore_label": IGNORE_INDEX,
    },
    "validation": {
        "per_class_iou": global_ious,
        "miou": global_miou,
        "distance_bands": band_results,
        "validation_time_seconds": float(
            validation_time
        ),
    },
}

RESULTS_JSON = (
    RESULTS_DIR
    / "validation_results.json"
)

with open(RESULTS_JSON, "w") as f:
    json.dump(
        results,
        f,
        indent=2
    )

# ---------------------------------------------------------------------------
# 14. SAVE HUMAN-READABLE REPORT
# ---------------------------------------------------------------------------

REPORT_PATH = (
    RESULTS_DIR
    / "VALIDATION_REPORT.txt"
)

with open(REPORT_PATH, "w") as f:

    f.write(
        "SIH26053 — STRONG MODEL VALIDATION REPORT\n"
    )
    f.write("=" * 75 + "\n\n")

    f.write("MODEL\n")
    f.write("-" * 75 + "\n")
    f.write(
        f"Model              : "
        f"{checkpoint['model_choice']}\n"
    )
    f.write(
        f"Architecture       : "
        f"StrongPointNetPPSegmenter\n"
    )
    f.write(
        f"Parameters         : "
        f"{parameter_count:,}\n"
    )
    f.write(
        f"Checkpoint epoch   : "
        f"{checkpoint['epoch']}\n"
    )
    f.write(
        f"Training loss      : "
        f"{checkpoint['loss']:.6f}\n"
    )
    f.write(
        f"Input features     : "
        f"x, y, z, intensity, range\n"
    )
    f.write(
        f"Classes            : "
        f"{classes}\n\n"
    )

    f.write("DATASET\n")
    f.write("-" * 75 + "\n")
    f.write(
        f"Validation shards  : "
        f"{len(val_manifest)}\n"
    )
    f.write(
        f"Validation frames  : "
        f"{expected_frames}\n"
    )
    f.write(
        f"Frames evaluated   : "
        f"{frames_evaluated}\n"
    )
    f.write(
        f"Total points       : "
        f"{total_points:,}\n"
    )
    f.write(
        f"Valid points       : "
        f"{valid_points:,}\n"
    )
    f.write(
        f"Ignored points     : "
        f"{ignored_points:,}\n\n"
    )

    f.write("GLOBAL VALIDATION\n")
    f.write("-" * 75 + "\n")

    for cls in classes:

        value = global_ious[cls]

        if value is None:
            f.write(
                f"{cls:20s}: None\n"
            )
        else:
            f.write(
                f"{cls:20s}: "
                f"{value:.4f}\n"
            )

    f.write(
        f"\nOverall mIoU       : "
        f"{global_miou:.4f}\n\n"
    )

    f.write("DISTANCE BANDS\n")
    f.write("-" * 75 + "\n")

    for band_name, result in band_results.items():

        f.write(
            f"\n{band_name}\n"
        )

        for cls in classes:

            value = result[
                "per_class_iou"
            ][cls]

            if value is None:
                f.write(
                    f"  {cls:18s}: None\n"
                )
            else:
                f.write(
                    f"  {cls:18s}: "
                    f"{value:.4f}\n"
                )

        f.write(
            f"  mIoU              : "
            f"{result['miou']:.4f}\n"
        )

    f.write(
        "\nValidation time: "
        f"{validation_time:.2f} seconds\n"
    )

# ---------------------------------------------------------------------------
# 15. RELOAD AND VALIDATE SAVED JSON
# ---------------------------------------------------------------------------

with open(RESULTS_JSON, "r") as f:
    saved_results = json.load(f)

assert saved_results["dataset"]["frames_evaluated"] == 80
assert saved_results["dataset"]["total_points"] == total_points
assert saved_results["dataset"]["valid_points"] == valid_points
assert saved_results["dataset"]["ignored_points"] == ignored_points

assert saved_results["validation"]["miou"] is not None

for cls in classes:
    value = saved_results[
        "validation"
    ]["per_class_iou"][cls]

    if value is not None:
        assert np.isfinite(value)

for band_name in band_results:

    band_miou = saved_results[
        "validation"
    ]["distance_bands"][band_name]["miou"]

    assert band_miou is not None
    assert np.isfinite(band_miou)

print("\n" + "=" * 75)
print("✓ STRONG MODEL VALIDATION COMPLETE")
print("=" * 75)

print("✓ All 80 validation frames evaluated.")
print("✓ Five input features used.")
print("✓ Ignore label 255 excluded from IoU.")
print("✓ Global per-class IoU calculated.")
print("✓ Global mIoU calculated.")
print("✓ Distance-band IoU calculated.")
print("✓ Validation JSON saved.")
print("✓ Human-readable validation report saved.")
print("✓ Saved results reloaded and validated.")

print(f"\nResults:")
print(f"  {RESULTS_JSON}")
print(f"  {REPORT_PATH}")

print("\nCell 9 complete.")

SIH26053 — 09 VALIDATE STRONG MODEL

Dataset:
  Path   : /kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1
  Exists : True

Checkpoint:
  Path   : /kaggle/working/strong_model/checkpoints/best_model.pth
  Exists : True

Device:
  cuda
  GPU : Tesla T4

Checkpoint metadata:
  Model          : pointnet_pp_strong
  Best epoch     : 47
  Training loss  : 0.376258
  Seed           : 42
  Parameters     : 241,604

Classes:
  0: drivable
  1: non_drivable
  2: static_obstacle
  3: dynamic_object

Model:
  Architecture : StrongPointNetPPSegmenter
  Parameters   : 241,604
  Input        : 5 features
  Output       : 4 classes

Shape validation:
  Input shape  : (1, 300, 5)
  Output shape : (1, 300, 4)
  ✓ Five-feature input accepted.
  ✓ Four-class output produced.
  ✓ Output contains no NaN/Inf.

Validation manifest:
  Shards : 1
  Frames : 80

FULL STRONG-MODEL VALIDATION
  evaluated 10/80 frames
  evaluated 20/80 frames
  evaluated 30/80 frames
  evaluated 40/8

In [21]:
# ============================================================================
# SIH26053 — 10 STRONG MODEL INFERENCE
# Generate real validation prediction samples
# ============================================================================

import torch
import json
import numpy as np
from pathlib import Path

print("=" * 75)
print("SIH26053 — 10 STRONG MODEL INFERENCE")
print("=" * 75)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/strong_model/checkpoints/best_model.pth"
)

OUTPUT_DIR = Path(
    "/kaggle/working/strong_model/predictions"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDataset:")
print(f"  Path       : {DATASET_DIR}")
print(f"  Exists     : {DATASET_DIR.exists()}")

print("\nCheckpoint:")
print(f"  Path       : {CHECKPOINT_PATH}")
print(f"  Exists     : {CHECKPOINT_PATH.exists()}")

print("\nOutput:")
print(f"  Directory  : {OUTPUT_DIR}")
print(f"  Samples    : {N_SAMPLES}")

print("\nDevice:")
print(f"  {device}")
if device.type == "cuda":
    print(f"  GPU        : {torch.cuda.get_device_name(0)}")

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

# ---------------------------------------------------------------------------
# Load checkpoint metadata
# ---------------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

classes = checkpoint["classes"]

assert len(classes) == 4

print("\nCheckpoint metadata:")
print(f"  Model          : {checkpoint['model_choice']}")
print(f"  Best epoch     : {checkpoint['epoch']}")
print(f"  Training loss  : {checkpoint['loss']:.6f}")
print(f"  Seed           : {checkpoint['seed']}")
print(f"  Parameters     : {checkpoint['parameters']}")
print(f"  Input features : {checkpoint['input_features']}")

print("\nClasses:")
for i, cls in enumerate(classes):
    print(f"  {i}: {cls}")

# ---------------------------------------------------------------------------
# Recreate the EXACT strong architecture
# ---------------------------------------------------------------------------
# IMPORTANT:
# The constructor is:
#     StrongPointNetPPSegmenter(num_classes=4, in_channels=5)
#
# StrongPointNetPPSegmenter and StrongSetAbstraction must already be defined
# in an earlier cell of this notebook.

strong_model = StrongPointNetPPSegmenter(
    num_classes=4,
    in_channels=5
)

strong_model.load_state_dict(
    checkpoint["model_state_dict"]
)

strong_model = strong_model.to(device)
strong_model.eval()

# Validate parameter count
parameter_count = sum(
    p.numel()
    for p in strong_model.parameters()
    if p.requires_grad
)

assert parameter_count == checkpoint["parameters"]

print("\nModel:")
print("  Architecture : StrongPointNetPPSegmenter")
print(f"  Parameters   : {parameter_count}")
print("  Input        : 5 features")
print("  Output       : 4 classes")

# ---------------------------------------------------------------------------
# Load validation manifest
# ---------------------------------------------------------------------------

manifest_path = DATASET_DIR / "metadata" / "val_manifest.json"

assert manifest_path.exists(), f"Validation manifest missing: {manifest_path}"

val_manifest = json.load(
    open(manifest_path)
)

assert len(val_manifest) > 0

shard_name = val_manifest[0]["shard"]
shard_path = DATASET_DIR / "val" / shard_name

assert shard_path.exists(), f"Validation shard missing: {shard_path}"

data = np.load(shard_path)

frame_count = val_manifest[0]["frame_count"]

print("\nValidation:")
print(f"  Shard        : {shard_name}")
print(f"  Frames       : {frame_count}")
print(f"  Samples used: {min(N_SAMPLES, frame_count)}")

# ---------------------------------------------------------------------------
# Shape smoke test on a real frame
# ---------------------------------------------------------------------------

test_points_np = data["points_0"]

assert test_points_np.ndim == 2
assert test_points_np.shape[1] == 5

test_points = (
    torch.from_numpy(test_points_np)
    .float()
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():
    test_logits = strong_model(test_points)

assert test_logits.shape == (
    1,
    test_points_np.shape[0],
    4
)

assert torch.isfinite(test_logits).all()

print("\nReal-frame smoke test:")
print(f"  Input shape  : {tuple(test_points.shape)}")
print(f"  Output shape : {tuple(test_logits.shape)}")
print("  ✓ Five input features accepted.")
print("  ✓ Four output classes produced.")
print("  ✓ Logits contain no NaN/Inf.")

del test_points
del test_logits

if device.type == "cuda":
    torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# Generate prediction samples
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("GENERATING STRONG MODEL PREDICTION SAMPLES")
print("=" * 75)

generated_files = []

with torch.no_grad():

    for i in range(min(N_SAMPLES, frame_count)):

        points_np = data[f"points_{i}"]
        labels_np = data[f"labels_{i}"]

        assert points_np.ndim == 2
        assert points_np.shape[1] == 5
        assert labels_np.ndim == 1
        assert len(points_np) == len(labels_np)

        points = (
            torch.from_numpy(points_np)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        logits = strong_model(points)

        assert logits.shape == (
            1,
            points_np.shape[0],
            4
        )

        assert torch.isfinite(logits).all()

        logits = logits.squeeze(0)

        # Predicted class
        preds = logits.argmax(dim=-1)

        # Per-point confidence
        probabilities = torch.softmax(logits, dim=-1)

        confidences = probabilities.max(
            dim=-1
        ).values

        preds_np = preds.cpu().numpy()
        confidences_np = confidences.cpu().numpy()

        # Validate outputs
        assert preds_np.shape == (len(points_np),)
        assert confidences_np.shape == (len(points_np),)

        assert np.all(
            (preds_np >= 0) &
            (preds_np < len(classes))
        )

        assert np.isfinite(confidences_np).all()

        assert np.min(confidences_np) >= 0.0
        assert np.max(confidences_np) <= 1.0 + 1e-6

        predicted_names = np.array(
            [classes[int(p)] for p in preds_np]
        )

        output_path = (
            OUTPUT_DIR /
            f"sample_{i:04d}.npz"
        )

        np.savez_compressed(
            output_path,

            # Original five features:
            # x, y, z, intensity, range
            points=points_np,

            # Original validation labels:
            true_labels=labels_np,

            # Predicted class IDs:
            predicted_class_idx=preds_np,

            # Human-readable predicted classes:
            predicted_class_names=predicted_names,

            # Maximum softmax probability per point:
            confidence=confidences_np,
        )

        assert output_path.exists()
        assert output_path.stat().st_size > 0

        generated_files.append(output_path)

        print(
            f"  ✓ sample_{i:04d}.npz"
            f" | points={len(points_np):,}"
            f" | mean_conf={confidences_np.mean():.4f}"
            f" | min_conf={confidences_np.min():.4f}"
            f" | max_conf={confidences_np.max():.4f}"
        )

# ---------------------------------------------------------------------------
# Verify all generated files
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("INFERENCE SAMPLE SUMMARY")
print("=" * 75)

print(f"Samples generated : {len(generated_files)}")
print(f"Output directory  : {OUTPUT_DIR}")

assert len(generated_files) == min(
    N_SAMPLES,
    frame_count
)

for path in generated_files:

    saved = np.load(path)

    assert "points" in saved
    assert "true_labels" in saved
    assert "predicted_class_idx" in saved
    assert "predicted_class_names" in saved
    assert "confidence" in saved

    assert saved["points"].shape[1] == 5
    assert len(saved["points"]) == len(saved["true_labels"])
    assert len(saved["points"]) == len(saved["predicted_class_idx"])
    assert len(saved["points"]) == len(saved["confidence"])

print("\n✓ All prediction samples generated.")
print("✓ Five-feature point clouds saved.")
print("✓ Ground-truth labels saved.")
print("✓ Predicted class IDs saved.")
print("✓ Predicted class names saved.")
print("✓ Per-point confidence saved.")
print("✓ All output files verified.")

print("\n" + "=" * 75)
print("✓ STRONG MODEL INFERENCE COMPLETE")
print("=" * 75)

print("\nCell 10 complete.")

SIH26053 — 10 STRONG MODEL INFERENCE

Dataset:
  Path       : /kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1
  Exists     : True

Checkpoint:
  Path       : /kaggle/working/strong_model/checkpoints/best_model.pth
  Exists     : True

Output:
  Directory  : /kaggle/working/strong_model/predictions
  Samples    : 10

Device:
  cuda
  GPU        : Tesla T4

Checkpoint metadata:
  Model          : pointnet_pp_strong
  Best epoch     : 47
  Training loss  : 0.376258
  Seed           : 42
  Parameters     : 241604
  Input features : ['x', 'y', 'z', 'intensity', 'range']

Classes:
  0: drivable
  1: non_drivable
  2: static_obstacle
  3: dynamic_object

Model:
  Architecture : StrongPointNetPPSegmenter
  Parameters   : 241604
  Input        : 5 features
  Output       : 4 classes

Validation:
  Shard        : shard_000.npz
  Frames       : 80
  Samples used: 10

Real-frame smoke test:
  Input shape  : (1, 34463, 5)
  Output shape : (1, 34463, 4)
  ✓ Five inpu

In [22]:
# ============================================================================
# SIH26053 — 11 STRONG MODEL BENCHMARK
# ============================================================================

import torch
import json
import time
from pathlib import Path

print("=" * 75)
print("SIH26053 — 11 STRONG MODEL BENCHMARK")
print("=" * 75)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/strong_model/checkpoints/best_model.pth"
)

RESULTS_DIR = Path(
    "/kaggle/working/strong_model/results/benchmark_results"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_JSON = RESULTS_DIR / "benchmark_results.json"
REPORT_TXT = RESULTS_DIR / "BENCHMARK_REPORT.txt"

assert DATASET_DIR.exists()
assert CHECKPOINT_PATH.exists()

# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDataset:")
print(f"  Path       : {DATASET_DIR}")
print(f"  Exists     : {DATASET_DIR.exists()}")

print("\nCheckpoint:")
print(f"  Path       : {CHECKPOINT_PATH}")
print(f"  Exists     : {CHECKPOINT_PATH.exists()}")

print("\nDevice:")
print(f"  {device}")

if device.type == "cuda":
    print(f"  GPU        : {torch.cuda.get_device_name(0)}")
    print(f"  CUDA       : {torch.version.cuda}")

# ---------------------------------------------------------------------------
# Load checkpoint
# ---------------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

classes = checkpoint["classes"]

assert checkpoint["model_choice"] == "pointnet_pp_strong"
assert len(classes) == 4
assert checkpoint["parameters"] == 241604
assert checkpoint["input_features"] == [
    "x",
    "y",
    "z",
    "intensity",
    "range",
]

print("\nCheckpoint metadata:")
print(f"  Model          : {checkpoint['model_choice']}")
print(f"  Best epoch     : {checkpoint['epoch']}")
print(f"  Training loss  : {checkpoint['loss']:.6f}")
print(f"  Seed           : {checkpoint['seed']}")
print(f"  Parameters     : {checkpoint['parameters']}")
print(f"  Input features : {checkpoint['input_features']}")

# ---------------------------------------------------------------------------
# Recreate exact strong model
# ---------------------------------------------------------------------------

strong_model = StrongPointNetPPSegmenter(
    num_classes=4,
    in_channels=5
)

strong_model.load_state_dict(
    checkpoint["model_state_dict"]
)

strong_model = strong_model.to(device)
strong_model.eval()

parameter_count = sum(
    p.numel()
    for p in strong_model.parameters()
    if p.requires_grad
)

assert parameter_count == 241604

print("\nModel:")
print("  Architecture : StrongPointNetPPSegmenter")
print(f"  Parameters   : {parameter_count}")
print("  Input        : 5 features")
print("  Output       : 4 classes")

# ---------------------------------------------------------------------------
# Shape smoke test
# ---------------------------------------------------------------------------

test_points = torch.randn(
    1,
    300,
    5,
    device=device
)

with torch.no_grad():
    test_output = strong_model(test_points)

assert test_output.shape == (1, 300, 4)
assert torch.isfinite(test_output).all()

print("\nShape validation:")
print("  Input shape  : (1, 300, 5)")
print("  Output shape : (1, 300, 4)")
print("  ✓ Forward pass successful.")
print("  ✓ Output contains no NaN/Inf.")

del test_points
del test_output

if device.type == "cuda":
    torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# Benchmark configuration
# ---------------------------------------------------------------------------

WARMUP_TRIALS = 5
N_TRIALS = 50

POINT_COUNTS = [
    1_000,
    10_000,
    50_000,
    100_000,
]

print("\n" + "=" * 75)
print("STRONG MODEL CUDA BENCHMARK")
print("=" * 75)

print("\nConfiguration:")
print(f"  GPU             : {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}")
print(f"  Warmup trials   : {WARMUP_TRIALS}")
print(f"  Timed trials    : {N_TRIALS}")
print(f"  Point counts    : {POINT_COUNTS}")

# ---------------------------------------------------------------------------
# Run benchmark
# ---------------------------------------------------------------------------

results = {}

with torch.no_grad():

    for n in POINT_COUNTS:

        # Five features:
        # x, y, z, intensity, range
        dummy = torch.randn(
            1,
            n,
            5,
            device=device
        )

        # ---------------------------------------------------------------
        # Warm-up
        # ---------------------------------------------------------------

        for _ in range(WARMUP_TRIALS):
            _ = strong_model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        # ---------------------------------------------------------------
        # Timed inference
        # ---------------------------------------------------------------

        start = time.perf_counter()

        for _ in range(N_TRIALS):
            _ = strong_model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        ms_per_frame = (
            1000.0 *
            elapsed /
            N_TRIALS
        )

        fps = (
            1000.0 /
            ms_per_frame
        )

        if device.type == "cuda":
            peak_memory_mb = (
                torch.cuda.max_memory_allocated()
                / (1024 ** 2)
            )
        else:
            peak_memory_mb = None

        assert np.isfinite(ms_per_frame) if "np" in globals() else True
        assert ms_per_frame > 0
        assert fps > 0

        results[str(n)] = {
            "points": n,
            "latency_ms_per_frame": float(ms_per_frame),
            "fps": float(fps),
            "peak_gpu_memory_mb": (
                float(peak_memory_mb)
                if peak_memory_mb is not None
                else None
            ),
        }

        if peak_memory_mb is not None:

            print(
                f"{n:>7,} points : "
                f"{ms_per_frame:8.3f} ms/frame | "
                f"{fps:8.2f} FPS | "
                f"peak={peak_memory_mb:8.2f} MB"
            )

        else:

            print(
                f"{n:>7,} points : "
                f"{ms_per_frame:8.3f} ms/frame | "
                f"{fps:8.2f} FPS"
            )

        del dummy

        if device.type == "cuda":
            torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# Save JSON
# ---------------------------------------------------------------------------

benchmark_metadata = {
    "model": checkpoint["model_choice"],
    "architecture": "StrongPointNetPPSegmenter",
    "parameters": parameter_count,
    "input_features": checkpoint["input_features"],
    "checkpoint_epoch": checkpoint["epoch"],
    "training_loss": checkpoint["loss"],
    "seed": checkpoint["seed"],
    "device": str(device),
    "gpu": (
        torch.cuda.get_device_name(0)
        if device.type == "cuda"
        else None
    ),
    "cuda_version": (
        torch.version.cuda
        if device.type == "cuda"
        else None
    ),
    "warmup_trials": WARMUP_TRIALS,
    "timed_trials": N_TRIALS,
    "results": results,
}

with open(RESULTS_JSON, "w") as f:
    json.dump(
        benchmark_metadata,
        f,
        indent=2
    )

# ---------------------------------------------------------------------------
# Human-readable report
# ---------------------------------------------------------------------------

with open(REPORT_TXT, "w") as f:

    f.write(
        "SIH26053 — STRONG MODEL BENCHMARK REPORT\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write(
        f"Model: StrongPointNetPPSegmenter\n"
    )
    f.write(
        f"Checkpoint epoch: {checkpoint['epoch']}\n"
    )
    f.write(
        f"Training loss: {checkpoint['loss']:.6f}\n"
    )
    f.write(
        f"Parameters: {parameter_count}\n"
    )
    f.write(
        f"Input features: {checkpoint['input_features']}\n"
    )
    f.write(
        f"Device: {device}\n"
    )

    if device.type == "cuda":
        f.write(
            f"GPU: {torch.cuda.get_device_name(0)}\n"
        )
        f.write(
            f"CUDA: {torch.version.cuda}\n"
        )

    f.write("\nBenchmark configuration:\n")
    f.write(
        f"Warmup trials: {WARMUP_TRIALS}\n"
    )
    f.write(
        f"Timed trials: {N_TRIALS}\n"
    )

    f.write("\nResults:\n")

    for n in POINT_COUNTS:

        r = results[str(n)]

        f.write(
            f"{n:>7,} points | "
            f"{r['latency_ms_per_frame']:.3f} ms/frame | "
            f"{r['fps']:.2f} FPS"
        )

        if r["peak_gpu_memory_mb"] is not None:
            f.write(
                f" | {r['peak_gpu_memory_mb']:.2f} MB"
            )

        f.write("\n")

    f.write(
        "\nIMPORTANT: Measurements are for the Kaggle GPU only.\n"
    )
    f.write(
        "They do not represent live-demo laptop performance.\n"
    )

# ---------------------------------------------------------------------------
# Reload and validate saved results
# ---------------------------------------------------------------------------

with open(RESULTS_JSON, "r") as f:
    reloaded = json.load(f)

assert reloaded["model"] == "pointnet_pp_strong"
assert reloaded["architecture"] == "StrongPointNetPPSegmenter"
assert reloaded["parameters"] == 241604
assert len(reloaded["results"]) == len(POINT_COUNTS)

for n in POINT_COUNTS:

    r = reloaded["results"][str(n)]

    assert r["points"] == n
    assert r["latency_ms_per_frame"] > 0
    assert r["fps"] > 0

    if r["peak_gpu_memory_mb"] is not None:
        assert r["peak_gpu_memory_mb"] > 0

print("\n" + "=" * 75)
print("STRONG MODEL BENCHMARK COMPLETE")
print("=" * 75)

print("\nArtifacts:")
print(f"  ✓ {RESULTS_JSON}")
print(f"  ✓ {REPORT_TXT}")

print("\n✓ All requested point counts benchmarked.")
print("✓ Warm-up completed before timing.")
print("✓ CUDA synchronization used where available.")
print("✓ Latency values validated.")
print("✓ FPS values validated.")
print("✓ Peak GPU memory values validated.")
print("✓ Benchmark JSON saved.")
print("✓ Human-readable benchmark report saved.")
print("✓ Saved results reloaded and validated.")

print("\nCell 11 complete.")

SIH26053 — 11 STRONG MODEL BENCHMARK

Dataset:
  Path       : /kaggle/input/datasets/akhiladogga/fovealidar-nuscenes-v1/fovealidar_nuscenes_v1
  Exists     : True

Checkpoint:
  Path       : /kaggle/working/strong_model/checkpoints/best_model.pth
  Exists     : True

Device:
  cuda
  GPU        : Tesla T4
  CUDA       : 12.8

Checkpoint metadata:
  Model          : pointnet_pp_strong
  Best epoch     : 47
  Training loss  : 0.376258
  Seed           : 42
  Parameters     : 241604
  Input features : ['x', 'y', 'z', 'intensity', 'range']

Model:
  Architecture : StrongPointNetPPSegmenter
  Parameters   : 241604
  Input        : 5 features
  Output       : 4 classes

Shape validation:
  Input shape  : (1, 300, 5)
  Output shape : (1, 300, 4)
  ✓ Forward pass successful.
  ✓ Output contains no NaN/Inf.

STRONG MODEL CUDA BENCHMARK

Configuration:
  GPU             : Tesla T4
  Warmup trials   : 5
  Timed trials    : 50
  Point counts    : [1000, 10000, 50000, 100000]
  1,000 points :  134.

In [23]:
# ============================================================================
# SIH26053 — 12 STRONG MODEL VISUALIZATIONS
# ============================================================================

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 75)
print("SIH26053 — 12 STRONG MODEL VISUALIZATIONS")
print("=" * 75)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

PREDICTIONS_DIR = Path(
    "/kaggle/working/strong_model/predictions"
)

VIS_DIR = Path(
    "/kaggle/working/strong_model/results/visualizations"
)

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PREDICTIONS_DIR.exists(), (
    f"Prediction directory not found: {PREDICTIONS_DIR}"
)

sample_files = sorted(
    PREDICTIONS_DIR.glob("sample_*.npz")
)

assert len(sample_files) >= 1, (
    "No prediction samples found."
)

print("\nPrediction samples:")
print(f"  Directory : {PREDICTIONS_DIR}")
print(f"  Found     : {len(sample_files)}")

# ---------------------------------------------------------------------------
# Class configuration
# ---------------------------------------------------------------------------

classes = [
    "drivable",
    "non_drivable",
    "static_obstacle",
    "dynamic_object",
]

CLASS_TO_IDX = {
    name: i
    for i, name in enumerate(classes)
}

# ---------------------------------------------------------------------------
# Visualization configuration
# ---------------------------------------------------------------------------

# We use x-y BEV because it is the most useful representation
# for LiDAR scene segmentation.
#
# Maximum number of points plotted per frame to keep PNG sizes reasonable.
MAX_PLOT_POINTS = 15000

# Representative samples.
# If fewer than 10 exist, automatically use all available samples.
sample_indices = list(
    range(min(10, len(sample_files)))
)

print("\nVisualization configuration:")
print(f"  Samples to visualize : {len(sample_indices)}")
print(f"  Maximum points/plot  : {MAX_PLOT_POINTS}")
print("  View                  : BEV (x-y)")
print("  Outputs               : PNG")

# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def choose_points(points, max_points=15000):
    """
    Deterministically subsample points for visualization.
    The original NPZ files remain untouched.
    """

    n = len(points)

    if n <= max_points:
        return np.arange(n)

    # Deterministic evenly spaced selection.
    return np.linspace(
        0,
        n - 1,
        max_points,
        dtype=np.int64
    )


def class_colors():
    """
    Fixed semantic colors for consistent visual comparison.
    """

    return {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:red",
        3: "tab:green",
    }


COLORS = class_colors()

# ---------------------------------------------------------------------------
# Generate visualizations
# ---------------------------------------------------------------------------

generated_files = []

for sample_idx in sample_indices:

    sample_path = sample_files[sample_idx]

    data = np.load(sample_path)

    points = data["points"]
    true_labels = data["true_labels"]
    predicted_labels = data["predicted_class_idx"]
    confidence = data["confidence"]

    assert points.ndim == 2
    assert points.shape[1] == 5

    assert len(true_labels) == len(points)
    assert len(predicted_labels) == len(points)
    assert len(confidence) == len(points)

    # ---------------------------------------------------------------
    # Stored features:
    # x, y, z, intensity, range
    # ---------------------------------------------------------------

    x = points[:, 0]
    y = points[:, 1]

    # Ignore label 255 is not plotted as a semantic class.
    valid_mask = true_labels != 255

    # Deterministic visualization subset.
    indices = choose_points(
        points,
        MAX_PLOT_POINTS
    )

    x_plot = x[indices]
    y_plot = y[indices]

    true_plot = true_labels[indices]
    pred_plot = predicted_labels[indices]
    conf_plot = confidence[indices]

    # ---------------------------------------------------------------
    # Figure 1:
    # Ground truth
    # ---------------------------------------------------------------

    fig = plt.figure(
        figsize=(10, 8)
    )

    for class_idx in range(len(classes)):

        mask = (
            true_plot == class_idx
        )

        if np.any(mask):

            plt.scatter(
                x_plot[mask],
                y_plot[mask],
                s=2,
                alpha=0.65,
                c=COLORS[class_idx],
                label=classes[class_idx],
                linewidths=0,
            )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — Ground Truth — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.legend(
        markerscale=4,
        fontsize=8,
        loc="upper right",
    )
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_ground_truth.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Figure 2:
    # Prediction
    # ---------------------------------------------------------------

    fig = plt.figure(
        figsize=(10, 8)
    )

    for class_idx in range(len(classes)):

        mask = (
            pred_plot == class_idx
        )

        if np.any(mask):

            plt.scatter(
                x_plot[mask],
                y_plot[mask],
                s=2,
                alpha=0.65,
                c=COLORS[class_idx],
                label=classes[class_idx],
                linewidths=0,
            )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — Prediction — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.legend(
        markerscale=4,
        fontsize=8,
        loc="upper right",
    )
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_prediction.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Figure 3:
    # Confidence
    # ---------------------------------------------------------------

    fig = plt.figure(
        figsize=(10, 8)
    )

    scatter = plt.scatter(
        x_plot,
        y_plot,
        s=2,
        alpha=0.7,
        c=conf_plot,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        linewidths=0,
    )

    plt.colorbar(
        scatter,
        label="Prediction confidence"
    )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — Prediction Confidence — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_confidence.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Figure 4:
    # Range
    # ---------------------------------------------------------------

    ranges_plot = points[indices, 4]

    fig = plt.figure(
        figsize=(10, 8)
    )

    scatter = plt.scatter(
        x_plot,
        y_plot,
        s=2,
        alpha=0.7,
        c=ranges_plot,
        cmap="turbo",
        linewidths=0,
    )

    plt.colorbar(
        scatter,
        label="LiDAR range (m)"
    )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — LiDAR Range — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_range.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Progress
    # ---------------------------------------------------------------

    print(
        f"  ✓ sample_{sample_idx:04d} | "
        f"points={len(points):,} | "
        f"plots=4"
    )

# ---------------------------------------------------------------------------
# Create visualization summary
# ---------------------------------------------------------------------------

summary = {
    "model": "pointnet_pp_strong",
    "architecture": "StrongPointNetPPSegmenter",
    "checkpoint_epoch": 47,
    "training_loss": 0.376258,
    "parameters": 241604,
    "input_features": [
        "x",
        "y",
        "z",
        "intensity",
        "range",
    ],
    "classes": classes,
    "samples_visualized": len(sample_indices),
    "max_points_per_plot": MAX_PLOT_POINTS,
    "view": "BEV x-y",
    "plot_types": [
        "ground_truth",
        "prediction",
        "confidence",
        "range",
    ],
    "generated_files": [
        str(p)
        for p in generated_files
    ],
}

SUMMARY_JSON = VIS_DIR / "VISUALIZATION_SUMMARY.json"

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

png_files = sorted(
    VIS_DIR.glob("*.png")
)

assert len(png_files) == (
    len(sample_indices) * 4
)

assert SUMMARY_JSON.exists()

# Check that every PNG has non-zero size.
for png in png_files:
    assert png.stat().st_size > 0

print("\n" + "=" * 75)
print("VISUALIZATION SUMMARY")
print("=" * 75)

print(f"Samples visualized : {len(sample_indices)}")
print(f"PNG files created  : {len(png_files)}")
print(f"Output directory   : {VIS_DIR}")

print("\nVisualization types:")
print("  ✓ Ground-truth BEV")
print("  ✓ Prediction BEV")
print("  ✓ Prediction confidence")
print("  ✓ LiDAR range")

print("\nArtifact validation:")
print("  ✓ All PNG files exist.")
print("  ✓ All PNG files have non-zero size.")
print("  ✓ Visualization summary saved.")
print("  ✓ Original prediction NPZ files were not modified.")

print("\n" + "=" * 75)
print("✓ STRONG MODEL VISUALIZATION COMPLETE")
print("=" * 75)

print("\nCell 12 complete.")

SIH26053 — 12 STRONG MODEL VISUALIZATIONS

Prediction samples:
  Directory : /kaggle/working/strong_model/predictions
  Found     : 10

Visualization configuration:
  Samples to visualize : 10
  Maximum points/plot  : 15000
  View                  : BEV (x-y)
  Outputs               : PNG
  ✓ sample_0000 | points=34,463 | plots=4
  ✓ sample_0001 | points=34,719 | plots=4
  ✓ sample_0002 | points=34,687 | plots=4
  ✓ sample_0003 | points=34,719 | plots=4
  ✓ sample_0004 | points=34,717 | plots=4
  ✓ sample_0005 | points=34,685 | plots=4
  ✓ sample_0006 | points=34,751 | plots=4
  ✓ sample_0007 | points=34,718 | plots=4
  ✓ sample_0008 | points=34,720 | plots=4
  ✓ sample_0009 | points=34,720 | plots=4

VISUALIZATION SUMMARY
Samples visualized : 10
PNG files created  : 40
Output directory   : /kaggle/working/strong_model/results/visualizations

Visualization types:
  ✓ Ground-truth BEV
  ✓ Prediction BEV
  ✓ Prediction confidence
  ✓ LiDAR range

Artifact validation:
  ✓ All PNG files ex

In [24]:
# ============================================================================
# SIH26053 — 12 STRONG MODEL VISUALIZATIONS
# ============================================================================

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 75)
print("SIH26053 — 12 STRONG MODEL VISUALIZATIONS")
print("=" * 75)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

PREDICTIONS_DIR = Path(
    "/kaggle/working/strong_model/predictions"
)

VIS_DIR = Path(
    "/kaggle/working/strong_model/results/visualizations"
)

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PREDICTIONS_DIR.exists(), (
    f"Prediction directory not found: {PREDICTIONS_DIR}"
)

sample_files = sorted(
    PREDICTIONS_DIR.glob("sample_*.npz")
)

assert len(sample_files) >= 1, (
    "No prediction samples found."
)

print("\nPrediction samples:")
print(f"  Directory : {PREDICTIONS_DIR}")
print(f"  Found     : {len(sample_files)}")

# ---------------------------------------------------------------------------
# Class configuration
# ---------------------------------------------------------------------------

classes = [
    "drivable",
    "non_drivable",
    "static_obstacle",
    "dynamic_object",
]

CLASS_TO_IDX = {
    name: i
    for i, name in enumerate(classes)
}

# ---------------------------------------------------------------------------
# Visualization configuration
# ---------------------------------------------------------------------------

# We use x-y BEV because it is the most useful representation
# for LiDAR scene segmentation.
#
# Maximum number of points plotted per frame to keep PNG sizes reasonable.
MAX_PLOT_POINTS = 15000

# Representative samples.
# If fewer than 10 exist, automatically use all available samples.
sample_indices = list(
    range(min(10, len(sample_files)))
)

print("\nVisualization configuration:")
print(f"  Samples to visualize : {len(sample_indices)}")
print(f"  Maximum points/plot  : {MAX_PLOT_POINTS}")
print("  View                  : BEV (x-y)")
print("  Outputs               : PNG")

# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def choose_points(points, max_points=15000):
    """
    Deterministically subsample points for visualization.
    The original NPZ files remain untouched.
    """

    n = len(points)

    if n <= max_points:
        return np.arange(n)

    # Deterministic evenly spaced selection.
    return np.linspace(
        0,
        n - 1,
        max_points,
        dtype=np.int64
    )


def class_colors():
    """
    Fixed semantic colors for consistent visual comparison.
    """

    return {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:red",
        3: "tab:green",
    }


COLORS = class_colors()

# ---------------------------------------------------------------------------
# Generate visualizations
# ---------------------------------------------------------------------------

generated_files = []

for sample_idx in sample_indices:

    sample_path = sample_files[sample_idx]

    data = np.load(sample_path)

    points = data["points"]
    true_labels = data["true_labels"]
    predicted_labels = data["predicted_class_idx"]
    confidence = data["confidence"]

    assert points.ndim == 2
    assert points.shape[1] == 5

    assert len(true_labels) == len(points)
    assert len(predicted_labels) == len(points)
    assert len(confidence) == len(points)

    # ---------------------------------------------------------------
    # Stored features:
    # x, y, z, intensity, range
    # ---------------------------------------------------------------

    x = points[:, 0]
    y = points[:, 1]

    # Ignore label 255 is not plotted as a semantic class.
    valid_mask = true_labels != 255

    # Deterministic visualization subset.
    indices = choose_points(
        points,
        MAX_PLOT_POINTS
    )

    x_plot = x[indices]
    y_plot = y[indices]

    true_plot = true_labels[indices]
    pred_plot = predicted_labels[indices]
    conf_plot = confidence[indices]

    # ---------------------------------------------------------------
    # Figure 1:
    # Ground truth
    # ---------------------------------------------------------------

    fig = plt.figure(
        figsize=(10, 8)
    )

    for class_idx in range(len(classes)):

        mask = (
            true_plot == class_idx
        )

        if np.any(mask):

            plt.scatter(
                x_plot[mask],
                y_plot[mask],
                s=2,
                alpha=0.65,
                c=COLORS[class_idx],
                label=classes[class_idx],
                linewidths=0,
            )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — Ground Truth — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.legend(
        markerscale=4,
        fontsize=8,
        loc="upper right",
    )
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_ground_truth.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Figure 2:
    # Prediction
    # ---------------------------------------------------------------

    fig = plt.figure(
        figsize=(10, 8)
    )

    for class_idx in range(len(classes)):

        mask = (
            pred_plot == class_idx
        )

        if np.any(mask):

            plt.scatter(
                x_plot[mask],
                y_plot[mask],
                s=2,
                alpha=0.65,
                c=COLORS[class_idx],
                label=classes[class_idx],
                linewidths=0,
            )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — Prediction — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.legend(
        markerscale=4,
        fontsize=8,
        loc="upper right",
    )
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_prediction.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Figure 3:
    # Confidence
    # ---------------------------------------------------------------

    fig = plt.figure(
        figsize=(10, 8)
    )

    scatter = plt.scatter(
        x_plot,
        y_plot,
        s=2,
        alpha=0.7,
        c=conf_plot,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        linewidths=0,
    )

    plt.colorbar(
        scatter,
        label="Prediction confidence"
    )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — Prediction Confidence — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_confidence.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Figure 4:
    # Range
    # ---------------------------------------------------------------

    ranges_plot = points[indices, 4]

    fig = plt.figure(
        figsize=(10, 8)
    )

    scatter = plt.scatter(
        x_plot,
        y_plot,
        s=2,
        alpha=0.7,
        c=ranges_plot,
        cmap="turbo",
        linewidths=0,
    )

    plt.colorbar(
        scatter,
        label="LiDAR range (m)"
    )

    plt.xlabel("x (m)")
    plt.ylabel("y (m)")
    plt.title(
        f"Strong Model — LiDAR Range — Sample {sample_idx:04d}"
    )
    plt.axis("equal")
    plt.grid(
        alpha=0.2
    )
    plt.tight_layout()

    output = (
        VIS_DIR /
        f"sample_{sample_idx:04d}_range.png"
    )

    plt.savefig(
        output,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    generated_files.append(output)

    # ---------------------------------------------------------------
    # Progress
    # ---------------------------------------------------------------

    print(
        f"  ✓ sample_{sample_idx:04d} | "
        f"points={len(points):,} | "
        f"plots=4"
    )

# ---------------------------------------------------------------------------
# Create visualization summary
# ---------------------------------------------------------------------------

summary = {
    "model": "pointnet_pp_strong",
    "architecture": "StrongPointNetPPSegmenter",
    "checkpoint_epoch": 47,
    "training_loss": 0.376258,
    "parameters": 241604,
    "input_features": [
        "x",
        "y",
        "z",
        "intensity",
        "range",
    ],
    "classes": classes,
    "samples_visualized": len(sample_indices),
    "max_points_per_plot": MAX_PLOT_POINTS,
    "view": "BEV x-y",
    "plot_types": [
        "ground_truth",
        "prediction",
        "confidence",
        "range",
    ],
    "generated_files": [
        str(p)
        for p in generated_files
    ],
}

SUMMARY_JSON = VIS_DIR / "VISUALIZATION_SUMMARY.json"

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

png_files = sorted(
    VIS_DIR.glob("*.png")
)

assert len(png_files) == (
    len(sample_indices) * 4
)

assert SUMMARY_JSON.exists()

# Check that every PNG has non-zero size.
for png in png_files:
    assert png.stat().st_size > 0

print("\n" + "=" * 75)
print("VISUALIZATION SUMMARY")
print("=" * 75)

print(f"Samples visualized : {len(sample_indices)}")
print(f"PNG files created  : {len(png_files)}")
print(f"Output directory   : {VIS_DIR}")

print("\nVisualization types:")
print("  ✓ Ground-truth BEV")
print("  ✓ Prediction BEV")
print("  ✓ Prediction confidence")
print("  ✓ LiDAR range")

print("\nArtifact validation:")
print("  ✓ All PNG files exist.")
print("  ✓ All PNG files have non-zero size.")
print("  ✓ Visualization summary saved.")
print("  ✓ Original prediction NPZ files were not modified.")

print("\n" + "=" * 75)
print("✓ STRONG MODEL VISUALIZATION COMPLETE")
print("=" * 75)

print("\nCell 12 complete.")

SIH26053 — 12 STRONG MODEL VISUALIZATIONS

Prediction samples:
  Directory : /kaggle/working/strong_model/predictions
  Found     : 10

Visualization configuration:
  Samples to visualize : 10
  Maximum points/plot  : 15000
  View                  : BEV (x-y)
  Outputs               : PNG
  ✓ sample_0000 | points=34,463 | plots=4
  ✓ sample_0001 | points=34,719 | plots=4
  ✓ sample_0002 | points=34,687 | plots=4
  ✓ sample_0003 | points=34,719 | plots=4
  ✓ sample_0004 | points=34,717 | plots=4
  ✓ sample_0005 | points=34,685 | plots=4
  ✓ sample_0006 | points=34,751 | plots=4
  ✓ sample_0007 | points=34,718 | plots=4
  ✓ sample_0008 | points=34,720 | plots=4
  ✓ sample_0009 | points=34,720 | plots=4

VISUALIZATION SUMMARY
Samples visualized : 10
PNG files created  : 40
Output directory   : /kaggle/working/strong_model/results/visualizations

Visualization types:
  ✓ Ground-truth BEV
  ✓ Prediction BEV
  ✓ Prediction confidence
  ✓ LiDAR range

Artifact validation:
  ✓ All PNG files ex

In [25]:
# ===========================================================================
# SIH26053 — 13 FINAL STRONG MODEL EXPORT + COMPLETE ARTIFACT ZIP
# ===========================================================================

import json
import shutil
import zipfile
import platform
from pathlib import Path
from datetime import datetime, timezone

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------

STRONG_ROOT = Path("/kaggle/working/strong_model")

CHECKPOINT_PATH = STRONG_ROOT / "checkpoints" / "best_model.pth"
VALIDATION_DIR = STRONG_ROOT / "results" / "validation_results"
BENCHMARK_DIR = STRONG_ROOT / "results" / "benchmark_results"
VISUALIZATION_DIR = STRONG_ROOT / "results" / "visualizations"
PREDICTION_DIR = STRONG_ROOT / "predictions"
RESULTS_DIR = STRONG_ROOT / "results"

EXPORT_DIR = STRONG_ROOT / "export"
ZIP_PATH = Path("/kaggle/working/SIH26053_STRONG_MODEL_COMPLETE.zip")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("SIH26053 — 13 FINAL STRONG MODEL EXPORT + COMPLETE ARTIFACT ZIP")
print("=" * 75)

# ---------------------------------------------------------------------------
# 1. VALIDATE REQUIRED ARTIFACTS
# ---------------------------------------------------------------------------

print("\nValidating strong-model artifacts...")

required_paths = {
    "checkpoint": CHECKPOINT_PATH,
    "validation_results": VALIDATION_DIR,
    "benchmark_results": BENCHMARK_DIR,
    "visualizations": VISUALIZATION_DIR,
    "predictions": PREDICTION_DIR,
}

for name, path in required_paths.items():
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {name:<20}: {path}")
    assert exists, f"Required artifact missing: {path}"

# ---------------------------------------------------------------------------
# 2. LOAD CHECKPOINT METADATA
# ---------------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

classes = checkpoint.get(
    "classes",
    [
        "drivable",
        "non_drivable",
        "static_obstacle",
        "dynamic_object",
    ],
)

checkpoint_metadata = {
    "model_choice": checkpoint.get("model_choice", "pointnet_pp_strong"),
    "epoch": checkpoint.get("epoch"),
    "loss": checkpoint.get("loss"),
    "seed": checkpoint.get("seed", 42),
    "parameters": checkpoint.get("parameters", 241604),
    "input_features": checkpoint.get(
        "input_features",
        ["x", "y", "z", "intensity", "range"]
    ),
    "classes": classes,
}

print("\nCheckpoint metadata:")
print(f"  Model          : {checkpoint_metadata['model_choice']}")
print(f"  Best epoch     : {checkpoint_metadata['epoch']}")
print(f"  Training loss  : {checkpoint_metadata['loss']}")
print(f"  Parameters     : {checkpoint_metadata['parameters']}")
print(f"  Input features : {checkpoint_metadata['input_features']}")

# ---------------------------------------------------------------------------
# 3. CREATE MODEL EXPORT DIRECTORY
# ---------------------------------------------------------------------------

MODEL_EXPORT = EXPORT_DIR / "model"
RESULTS_EXPORT = EXPORT_DIR / "results"
PREDICTIONS_EXPORT = EXPORT_DIR / "predictions"
VISUALIZATIONS_EXPORT = EXPORT_DIR / "visualizations"

for directory in [
    MODEL_EXPORT,
    RESULTS_EXPORT,
    PREDICTIONS_EXPORT,
    VISUALIZATIONS_EXPORT,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 4. COPY MODEL CHECKPOINT
# ---------------------------------------------------------------------------

shutil.copy2(
    CHECKPOINT_PATH,
    MODEL_EXPORT / "best_model.pth"
)

print("\nModel artifacts:")
print("  ✓ best_model.pth")

# ---------------------------------------------------------------------------
# 5. COPY DATASET CONFIGURATION / MANIFESTS IF AVAILABLE
# ---------------------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/"
    "fovealid ar-nuscenes-v1/fovealid ar_nuscenes_v1"
)

# Correct known dataset path used by the successful cells.
DATASET_DIR = Path(
    "/kaggle/input/datasets/akhiladogga/"
    "fovealidar-nuscenes-v1/fovealidar_nuscenes_v1"
)

metadata_dir = DATASET_DIR / "metadata"

dataset_files = [
    "classes.json",
    "preprocessing_config.json",
    "train_manifest.json",
    "val_manifest.json",
]

print("\nDataset configuration artifacts:")

for filename in dataset_files:
    source = metadata_dir / filename

    if source.exists():
        shutil.copy2(
            source,
            MODEL_EXPORT / filename
        )
        print(f"  ✓ {filename}")
    else:
        print(f"  - {filename} not found")

# ---------------------------------------------------------------------------
# 6. CREATE MODEL METADATA
# ---------------------------------------------------------------------------

model_metadata = {
    "project": "SIH26053",
    "model_name": "pointnet_pp_strong",
    "architecture": "StrongPointNetPPSegmenter",

    "checkpoint": {
        "epoch": checkpoint_metadata["epoch"],
        "training_loss": checkpoint_metadata["loss"],
        "seed": checkpoint_metadata["seed"],
        "parameters": checkpoint_metadata["parameters"],
    },

    "input_features": checkpoint_metadata["input_features"],
    "num_input_features": len(checkpoint_metadata["input_features"]),

    "classes": classes,
    "num_classes": len(classes),

    "dataset": {
        "source": "nuScenes-mini",
        "version": "v1.0-mini",
        "max_range_m": 100.0,
    },

    "training": {
        "epochs_completed": 50,
        "batch_size": 4,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "class_weighted_cross_entropy": True,
        "ignore_index": -100,
        "original_ignore_label": 255,
    },

    "validation": {
        "frames": 80,
        "global_mIoU": 0.5906,
    },

    "benchmark": {
        "gpu": "Tesla T4",
        "cuda": "12.8",
    },

    "environment": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "platform": platform.platform(),
    },

    "exported_at": datetime.now(timezone.utc).isoformat(),
}

with open(
    MODEL_EXPORT / "MODEL_METADATA.json",
    "w"
) as f:
    json.dump(
        model_metadata,
        f,
        indent=2
    )

print("  ✓ MODEL_METADATA.json")

# ---------------------------------------------------------------------------
# 7. COPY ALL VALIDATION RESULTS
# ---------------------------------------------------------------------------

print("\nValidation artifacts:")

validation_export = RESULTS_EXPORT / "validation_results"
validation_export.mkdir(parents=True, exist_ok=True)

validation_count = 0

for source in VALIDATION_DIR.rglob("*"):
    if source.is_file():
        relative = source.relative_to(VALIDATION_DIR)
        destination = validation_export / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        validation_count += 1
        print(f"  ✓ {relative}")

# ---------------------------------------------------------------------------
# 8. COPY ALL BENCHMARK RESULTS
# ---------------------------------------------------------------------------

print("\nBenchmark artifacts:")

benchmark_export = RESULTS_EXPORT / "benchmark_results"
benchmark_export.mkdir(parents=True, exist_ok=True)

benchmark_count = 0

for source in BENCHMARK_DIR.rglob("*"):
    if source.is_file():
        relative = source.relative_to(BENCHMARK_DIR)
        destination = benchmark_export / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        benchmark_count += 1
        print(f"  ✓ {relative}")

# ---------------------------------------------------------------------------
# 9. COPY TRAINING HISTORY / CURVES / OTHER RESULT FILES
# ---------------------------------------------------------------------------

print("\nTraining/result artifacts:")

training_result_files = 0

for source in RESULTS_DIR.rglob("*"):
    if not source.is_file():
        continue

    # Already copied into dedicated folders.
    if VALIDATION_DIR in source.parents:
        continue

    if BENCHMARK_DIR in source.parents:
        continue

    # Do not copy the export directory into itself.
    if EXPORT_DIR in source.parents:
        continue

    relative = source.relative_to(RESULTS_DIR)

    destination = RESULTS_EXPORT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source, destination)

    training_result_files += 1
    print(f"  ✓ {relative}")

# ---------------------------------------------------------------------------
# 10. COPY ALL PREDICTION SAMPLES
# ---------------------------------------------------------------------------

print("\nPrediction samples:")

prediction_count = 0

for source in sorted(PREDICTION_DIR.rglob("*")):
    if source.is_file():
        relative = source.relative_to(PREDICTION_DIR)
        destination = PREDICTIONS_EXPORT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(
            source,
            destination
        )

        prediction_count += 1
        print(f"  ✓ {relative}")

print(f"\nPrediction files exported: {prediction_count}")

# ---------------------------------------------------------------------------
# 11. COPY ALL VISUALIZATIONS / GRAPHS / PNGS
# ---------------------------------------------------------------------------

print("\nVisualization / graph artifacts:")

visualization_count = 0

for source in sorted(VISUALIZATION_DIR.rglob("*")):
    if source.is_file():
        relative = source.relative_to(VISUALIZATION_DIR)

        destination = VISUALIZATIONS_EXPORT / relative
        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source,
            destination
        )

        visualization_count += 1
        print(f"  ✓ {relative}")

print(
    f"\nVisualization files exported: "
    f"{visualization_count}"
)

# ---------------------------------------------------------------------------
# 12. SEARCH FOR ANY OTHER USEFUL ARTIFACTS INSIDE STRONG_MODEL
# ---------------------------------------------------------------------------

print("\nScanning strong-model directory for additional artifacts...")

# Anything under strong_model that is not already part of export
# is recorded in the manifest. The final ZIP will include the complete
# strong_model export plus the original project artifacts separately
# where appropriate.

additional_artifacts = []

for source in STRONG_ROOT.rglob("*"):
    if not source.is_file():
        continue

    if EXPORT_DIR in source.parents:
        continue

    relative = source.relative_to(STRONG_ROOT)

    additional_artifacts.append({
        "path": str(relative),
        "size_bytes": source.stat().st_size,
        "size_mb": round(
            source.stat().st_size / (1024 ** 2),
            4
        ),
    })

print(
    f"Additional source artifacts found: "
    f"{len(additional_artifacts)}"
)

# ---------------------------------------------------------------------------
# 13. CREATE FINAL EXPORT MANIFEST
# ---------------------------------------------------------------------------

manifest = {
    "project": "SIH26053",

    "export_type": "complete_strong_model_export",

    "created_at": datetime.now(timezone.utc).isoformat(),

    "model": model_metadata,

    "artifact_counts": {
        "validation_files": validation_count,
        "benchmark_files": benchmark_count,
        "prediction_files": prediction_count,
        "visualization_files": visualization_count,
        "training_result_files": training_result_files,
        "additional_source_artifacts": len(additional_artifacts),
    },

    "validation_summary": {
        "frames": 80,
        "total_points": 2777340,
        "valid_points": 2076318,
        "ignored_points": 701022,

        "per_class_iou": {
            "drivable": 0.8461,
            "non_drivable": 0.5266,
            "static_obstacle": 0.6141,
            "dynamic_object": 0.3756,
        },

        "mIoU": 0.5906,

        "distance_band_mIoU": {
            "0-10m": 0.5732,
            "10-30m": 0.5305,
            "30-60m": 0.4108,
            "60-100m": 0.3069,
        },
    },

    "benchmark_summary": {
        "gpu": "Tesla T4",
        "warmup_trials": 5,
        "timed_trials": 50,

        "results": {
            "1000": {
                "ms_per_frame": 134.594,
                "fps": 7.43,
                "peak_memory_mb": 65.12,
            },
            "10000": {
                "ms_per_frame": 136.765,
                "fps": 7.31,
                "peak_memory_mb": 78.29,
            },
            "50000": {
                "ms_per_frame": 137.169,
                "fps": 7.29,
                "peak_memory_mb": 313.12,
            },
            "100000": {
                "ms_per_frame": 146.072,
                "fps": 6.85,
                "peak_memory_mb": 608.32,
            },
        },
    },

    "additional_source_artifacts": additional_artifacts,

    "export_structure": [
        "model/",
        "results/",
        "predictions/",
        "visualizations/",
        "EXPORT_MANIFEST.json",
        "README_EXPORT.txt",
    ],
}

with open(
    EXPORT_DIR / "EXPORT_MANIFEST.json",
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print("\n✓ EXPORT_MANIFEST.json created")

# ---------------------------------------------------------------------------
# 14. CREATE README FOR THE EXPORT
# ---------------------------------------------------------------------------

readme = f"""
SIH26053 — STRONG MODEL COMPLETE EXPORT
========================================

Model:
  StrongPointNetPPSegmenter
  Model choice: pointnet_pp_strong

Training:
  Epochs completed : 50
  Best epoch       : {checkpoint_metadata["epoch"]}
  Best loss        : {checkpoint_metadata["loss"]}
  Parameters       : {checkpoint_metadata["parameters"]}
  Input features   : x, y, z, intensity, range
  Classes          : 4

Validation:
  Frames           : 80
  Overall mIoU     : 0.5906

Per-class IoU:
  drivable         : 0.8461
  non_drivable     : 0.5266
  static_obstacle  : 0.6141
  dynamic_object   : 0.3756

Distance-band mIoU:
  0-10m            : 0.5732
  10-30m           : 0.5305
  30-60m           : 0.4108
  60-100m          : 0.3069

Benchmark:
  GPU              : Tesla T4
  1,000 points     : 134.594 ms / 7.43 FPS
  10,000 points    : 136.765 ms / 7.31 FPS
  50,000 points    : 137.169 ms / 7.29 FPS
  100,000 points   : 146.072 ms / 6.85 FPS

Included:
  - model checkpoint
  - model metadata
  - class configuration
  - preprocessing configuration
  - train/validation manifests
  - validation metrics
  - validation report
  - benchmark metrics
  - benchmark report
  - prediction samples
  - prediction confidence data
  - visualization PNGs
  - training history / result files
  - export manifest
  - this README

This package contains the complete strong-model artifact set
generated during the 50-epoch training/validation/inference/
benchmark/visualization pipeline.

Export generated:
  {datetime.now(timezone.utc).isoformat()}
"""

with open(
    EXPORT_DIR / "README_EXPORT.txt",
    "w"
) as f:
    f.write(readme.strip() + "\n")

print("✓ README_EXPORT.txt created")

# ---------------------------------------------------------------------------
# 15. CREATE ZIP
# ---------------------------------------------------------------------------

print("\nCreating final ZIP package...")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

files_added = 0
zip_size_before = 0

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for source in sorted(EXPORT_DIR.rglob("*")):

        if not source.is_file():
            continue

        relative = source.relative_to(EXPORT_DIR)

        zf.write(
            source,
            arcname=str(
                Path("SIH26053_STRONG_MODEL") / relative
            )
        )

        files_added += 1
        zip_size_before += source.stat().st_size

# ---------------------------------------------------------------------------
# 16. VERIFY ZIP
# ---------------------------------------------------------------------------

assert ZIP_PATH.exists(), "ZIP file was not created."
assert ZIP_PATH.stat().st_size > 0, "ZIP file is empty."

with zipfile.ZipFile(
    ZIP_PATH,
    mode="r"
) as zf:

    bad_file = zf.testzip()

    assert bad_file is None, (
        f"ZIP integrity check failed: {bad_file}"
    )

    zip_members = zf.namelist()

print("\nZIP verification:")
print(f"  ✓ ZIP exists")
print(f"  ✓ ZIP integrity verified")
print(f"  ✓ Files inside ZIP : {len(zip_members)}")

# ---------------------------------------------------------------------------
# 17. FINAL SUMMARY
# ---------------------------------------------------------------------------

zip_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
source_mb = zip_size_before / (1024 ** 2)

print("\n" + "=" * 75)
print("FINAL STRONG MODEL EXPORT SUMMARY")
print("=" * 75)

print(f"""
Model:
  Architecture       : StrongPointNetPPSegmenter
  Best epoch         : {checkpoint_metadata["epoch"]}
  Best training loss : {checkpoint_metadata["loss"]}
  Parameters         : {checkpoint_metadata["parameters"]}
  Input features     : 5
  Classes            : 4

Validation:
  Frames             : 80
  Overall mIoU       : 0.5906

Artifacts:
  Validation files   : {validation_count}
  Benchmark files    : {benchmark_count}
  Prediction files   : {prediction_count}
  Visualization PNGs : {visualization_count}
  Other result files : {training_result_files}

Package:
  Export directory   : {EXPORT_DIR}
  ZIP file           : {ZIP_PATH}
  ZIP size           : {zip_mb:.2f} MB
  Source artifact    : {source_mb:.2f} MB
  ZIP members        : {len(zip_members)}
""")

print("=" * 75)
print("✓ COMPLETE STRONG MODEL EXPORT CREATED")
print("=" * 75)
print("✓ Model checkpoint included.")
print("✓ Validation results included.")
print("✓ Benchmark results included.")
print("✓ Prediction samples included.")
print("✓ All visualization PNGs included.")
print("✓ Training/result artifacts included.")
print("✓ Metadata and manifests included.")
print("✓ Export manifest included.")
print("✓ ZIP integrity verified.")
print()
print(f"FINAL ZIP:")
print(ZIP_PATH)
print()
print("Cell 13 complete.")

SIH26053 — 13 FINAL STRONG MODEL EXPORT + COMPLETE ARTIFACT ZIP

Validating strong-model artifacts...
  ✓ checkpoint          : /kaggle/working/strong_model/checkpoints/best_model.pth
  ✓ validation_results  : /kaggle/working/strong_model/results/validation_results
  ✓ benchmark_results   : /kaggle/working/strong_model/results/benchmark_results
  ✓ visualizations      : /kaggle/working/strong_model/results/visualizations
  ✓ predictions         : /kaggle/working/strong_model/predictions

Checkpoint metadata:
  Model          : pointnet_pp_strong
  Best epoch     : 47
  Training loss  : 0.3762576359289664
  Parameters     : 241604
  Input features : ['x', 'y', 'z', 'intensity', 'range']

Model artifacts:
  ✓ best_model.pth

Dataset configuration artifacts:
  ✓ classes.json
  ✓ preprocessing_config.json
  ✓ train_manifest.json
  ✓ val_manifest.json
  ✓ MODEL_METADATA.json

Validation artifacts:
  ✓ VALIDATION_REPORT.txt
  ✓ validation_results.json

Benchmark artifacts:
  ✓ benchmark_resu